<div align="center">

# Head and shoulders, knees and toes

### A proven public 0.936 knee-MRI checkpoint



</div>

---

| Public LB | Findings | Inference families | Network | Proven source |
|:--:|:--:|:--:|:--:|:--:|
| **0.936** | **12** | **DINO + RadImageNet + dual CoAtNet** | **offline** | **DINOsaur V10** |

> This release is the reproducible control checkpoint for new competitors. It
> is a source-faithful Apache-2.0 port of [Roman Tamrazov's scored DINOsaur V10](https://www.kaggle.com/code/romantamrazov/rsna-knee-dinosaur-v4?scriptVersionId=344688577).
> Model, preprocessing, and fusion code is unchanged. Three owner-specific
> Kaggle mount strings are translated to the standard public-dataset mounts;
> only those paths, this explanatory markdown, and the H&S identity differ.

## What the checkpoint does

```text
RSNA DICOM studies
       |
       +-- DINOv2/DINOv3 protocol models -----------+
       |                                             |
       +-- RadImageNet diversity + calibration ------+--> cohort-rank fusion
       |                                             |
       +-- MaxSpan v5 + WideDense v4 CoAtNets -------+
                         target-specific complement          |
                                                            v
                                                     submission.csv
```

The V10 change is not a generic model soup. It uses MaxSpan v5 as the primary
Raptor checkpoint, WideDense v4 as a complementary checkpoint, and restricts
the complement to predeclared findings and weights before fusing with the
transformer/RadImageNet anchor.

## Reproducibility contract

- Proven public score: **0.936**.
- Proven source version: **`344688577`**.
- Upstream notebook source SHA-256: **`4504962c3b803a4385c2188084c13c3fe806ad006fd49b2201a03a78c10f460c`**.
- The four prediction cells match V10 except for three audited mount-path
  substitutions across the first two cells.
- Required inputs are kept exactly as declared by the scored source.
- Expected output: `/kaggle/working/submission.csv` with one ID column and the
  twelve official findings.
- This is a competition checkpoint, **not a clinical device**.

## Credit by retained responsibility

| Retained component | Public source |
|---|---|
| Scored V10 inference graph and dual-checkpoint target fusion | [Roman Tamrazov — DINOsaur V10](https://www.kaggle.com/code/romantamrazov/rsna-knee-dinosaur-v4?scriptVersionId=344688577) |
| DINO/RadImageNet reproduction assets | [Tony Li — RSNA Knee reproduction assets](https://www.kaggle.com/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets) |
| MaxSpan and WideDense Raptor checkpoints | [Dread Development — Raptor datasets](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-maxspan) |
| Standard DINOv2 ViT-S/14 base | [Meta Research — DINOv2](https://github.com/facebookresearch/dinov2) |

The notebook is packaged and audited here by **prvsiyan** so the exact public
checkpoint can be run, inspected, and extended from one place.

In [1]:
# MANUAL PROFILE SELECTOR - edit only this cell before Save Version.
#
# New signal:      v9_specialist, v9_gold_gated
# Lower variance:  direction_005, bootstrap_shrunk
# Old probes:      raptor065, clinical_moderate, hybrid_public
# Highest risk:    aggressive, aggressive_residual
FINAL_PROFILE = "clinical_moderate"
N_BOOTSTRAPS = 1000


In [2]:
"""Resolve Kaggle mounts without assuming UI or API directory layouts."""

import os
from pathlib import Path


def _input_root():
    return Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))


def _existing_directories(paths):
    seen = set()
    for path in paths:
        path = Path(path)
        try:
            key = str(path.resolve())
        except OSError:
            key = str(path)
        if key not in seen and path.is_dir():
            seen.add(key)
            yield path


def kaggle_data_roots(input_root=None):
    """Return dataset roots while excluding the very large competition tree."""
    root = Path(input_root or _input_root())
    candidates = []
    if root.is_dir():
        for child in root.iterdir():
            if not child.is_dir() or child.name.lower() in {"competitions", "models"}:
                continue
            if child.name.lower() == "datasets":
                for owner in child.iterdir():
                    if owner.is_dir():
                        candidates.extend(path for path in owner.iterdir() if path.is_dir())
            else:
                is_competition = (child / "train.csv").is_file() and (child / "test.csv").is_file()
                if not is_competition:
                    candidates.append(child)
    return list(_existing_directories(candidates))


def resolve_competition(slug, input_root=None):
    root = Path(input_root or _input_root())
    candidates = [
        root / "competitions" / slug,
        root / slug,
    ]
    competitions = root / "competitions"
    if competitions.is_dir():
        candidates.extend(
            path for path in competitions.iterdir() if path.is_dir() and path.name == slug
        )
    for path in _existing_directories(candidates):
        if (path / "train.csv").is_file() and (path / "test.csv").is_file():
            return path
    raise FileNotFoundError(
        f"competition {slug!r} was not found below {root}; "
        "attach it through Add Input > Competitions"
    )


def resolve_dataset(slug, owner=None, input_root=None):
    root = Path(input_root or _input_root())
    candidates = [root / slug, root / "datasets" / slug]
    if owner:
        candidates.insert(0, root / "datasets" / owner / slug)
    candidates.extend(
        path
        for path in kaggle_data_roots(root)
        if path.name.lower() == slug.lower()
    )
    resolved = list(_existing_directories(candidates))
    if resolved:
        return resolved[0]
    owner_hint = f"{owner}/" if owner else ""
    raise FileNotFoundError(
        f"dataset {owner_hint}{slug} was not found below {root}; attach it through Add Input"
    )


def resolve_dinov2_small(input_root=None):
    root = Path(input_root or _input_root())
    candidates = [
        root / "models" / "metaresearch" / "dinov2" / "pytorch" / "small" / "1",
        root / "models" / "metaresearch" / "dinov2" / "PyTorch" / "small" / "1",
        root / "dinov2" / "pytorch" / "small" / "1",
        root / "dinov2" / "PyTorch" / "small" / "1",
    ]
    models_root = root / "models"
    if models_root.is_dir():
        candidates.extend(
            path.parent
            for path in models_root.rglob("config.json")
            if "dinov2" in str(path).lower() and "small" in str(path).lower()
        )
    for path in _existing_directories(candidates):
        if (path / "config.json").is_file():
            return path
    raise FileNotFoundError(
        "DINOv2 small/1 config.json was not found; attach "
        "metaresearch/dinov2, PyTorch, small/1 through Add Input > Models"
    )


def find_input_files(filename, input_root=None, preferred=()):
    """Find an exact filename only inside mounted datasets, never competition images."""
    preferred = tuple(str(item).lower() for item in preferred)
    hits = []
    for data_root in kaggle_data_roots(input_root):
        direct = data_root / filename
        if direct.is_file():
            hits.append(direct)
        hits.extend(path for path in data_root.rglob(filename) if path.is_file())
    unique = {str(path.resolve()): path for path in hits}

    def priority(path):
        text = str(path).lower()
        preferred_rank = next(
            (index for index, token in enumerate(preferred) if token in text),
            len(preferred),
        )
        return preferred_rank, len(path.parts), text

    return sorted(unique.values(), key=priority)


def find_input_file(filename, input_root=None, preferred=(), required=True):
    hits = find_input_files(filename, input_root=input_root, preferred=preferred)
    if hits:
        return hits[0]
    if required:
        raise FileNotFoundError(
            f"{filename} was not found in mounted datasets below "
            f"{Path(input_root or _input_root())}"
        )
    return None


def resolve_rsna_inputs(input_root=None):
    root = Path(input_root or _input_root())
    resolved = {
        "input_root": root,
        "competition": resolve_competition("rsna-knee-abnormality-detection", root),
        "repro_assets": resolve_dataset(
            "rsna-knee-bend-dinov3-0917-repro-assets", "tonylica", root
        ),
        "dinov2": resolve_dinov2_small(root),
        "raptor_v5": find_input_file(
            "raptor_ft_coatnet_v5_full_swa.pt",
            root,
            preferred=("raptor-knee-maxspan",),
        ),
        "raptor_v4": find_input_file(
            "raptor_ft_coatnet_v4_full.pt",
            root,
            preferred=("raptor-knee-widedense",),
        ),
        "raptor_v9": find_input_file(
            "raptor_ft_coatnet_v9_full.pt",
            root,
            preferred=("raptor-knee-finespacing",),
            required=False,
        ),
    }
    return resolved


def print_rsna_preflight(resolved, selected_profile=None):
    print("Kaggle input preflight", flush=True)
    for key, path in resolved.items():
        print(f"  {key:14s}: {path if path is not None else 'MISSING'}", flush=True)
    if selected_profile in {"v9_specialist", "v9_gold_gated"} and resolved["raptor_v9"] is None:
        print(
            "  ACTION        : Add Input dreaddevelopment/raptor-knee-finespacing; "
            "without it the selected profile falls back to exact V10",
            flush=True,
        )

_RSNA_PATHS = resolve_rsna_inputs()
print_rsna_preflight(_RSNA_PATHS, FINAL_PROFILE)
_RSNA_COMP_ROOT = _RSNA_PATHS["competition"]
_RSNA_ASSET_ROOT = _RSNA_PATHS["repro_assets"]
_RSNA_DINO_ROOT = _RSNA_PATHS["dinov2"]


Kaggle input preflight
  input_root    : /kaggle/input
  competition   : /kaggle/input/competitions/rsna-knee-abnormality-detection
  repro_assets  : /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets
  dinov2        : /kaggle/input/models/metaresearch/dinov2/pytorch/small/1
  raptor_v5     : /kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan/raptor_ft_coatnet_v5_full_swa.pt
  raptor_v4     : /kaggle/input/datasets/dreaddevelopment/raptor-knee-widedense/raptor_ft_coatnet_v4_full.pt
  raptor_v9     : MISSING


In [3]:
from __future__ import annotations
import os as _os
def _comp_root():
    return str(_RSNA_COMP_ROOT)
_COMP_ROOT = _comp_root()
import os
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
ASSET = Path(_RSNA_ASSET_ROOT)
ROOT = Path(_RSNA_COMP_ROOT)
DINO = Path(_RSNA_DINO_ROOT)
T0 = time.time()
DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
SEED = 2026
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []

def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])
ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None

def build_cache(slot_map, plane_map, lat_map, tag):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f'{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    n_job = len(jobs)
    t_ord = time.time()
    n_slice_total = sum((len(j[3]['files']) for j in jobs))
    log(f'{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)')
    ok = done = 0
    CHUNK_O = 1024
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec['SeriesInstanceUID'])
            if e and len(e['files']) == len(rec['files']):
                rec['ordered'] = e['files']
                ok += int(e['good'])
                hit += 1
        jobs = [j for j in jobs if 'ordered' not in j[3]]
        log(f'{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read')
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(block, pool.map(lambda j: order_slices(j[3]), block)):
                rec['ordered'] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec['SeriesInstanceUID']] = {'files': files, 'good': bool(good)}
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f'{tag}: ordering budget spent at {done}/{len(jobs)}; the rest keep file order')
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix('.tmp')
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f'{tag}: ordered {ok}/{n_job} by geometry ({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: decoding {len(jobs)} slot-series')
    n_failed_before = len(DECODE_FAILED)
    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane, lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f'  {tag} {done}/{len(jobs)}')
            if time.time() - T0 > TIME_BUDGET:
                log(f'  {tag}: time budget reached during decode')
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f'{tag}: {int(mask.sum())}/{len(jobs)} slots filled' + (f'; {n_failed} series had a slice that would not decode' if n_failed else ''))
    gc.collect()
    return (studies, cache, mask)

class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)
FINGERPRINT_TOL = 0.002

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
LEGACY_FOLD_SOFTPOOL_BETA = {'ACL': 6.0, 'MCL': 6.0, 'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0, "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0}
LEGACY_FOLD_SOFTPOOL_ALPHA = {'ACL': 0.2, 'MCL': 0.2, 'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25, "Baker's": 0.2, 'Contusion': 0.2, 'Fracture': 0.15}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

def legacy_fold_soft_window_pool(original_probs, target_idx):
    values = original_probs.mean(0).clone()
    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():
        j = target_idx[target]
        x = original_probs[:, :, j]
        weight = torch.softmax(float(beta) * x, dim=0)
        values[:, j] = (weight * x).sum(0)
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out, public_soft_out = ([], [], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)
            public_soft_out.append(public_soft.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier, public_soft)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            state, fp = (m['state'], None)
        else:
            ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
            state, fp = (ck['model'], ck.get('fingerprint'))
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no stored fingerprint (legacy bundle) -- accepted at reduced weight")
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p, public_soft = predicted
    else:
        p, public_p, public_soft = (predicted, None, None)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def combine_public_members_by_fold(per_member, pred_key='pred'):
    all_ids = sorted({study for member in per_member for study in member['ids']})
    position = {study: i for i, study in enumerate(all_ids)}
    groups = {}
    for i, member in enumerate(per_member):
        fold = member.get('fold')
        key = f'fold_{fold}' if fold is not None else f'member_{i}'
        groups.setdefault(key, []).append(member)
    fold_ranks, diagnostics = ([], [])
    for key, members_in_fold in sorted(groups.items()):
        matrices = []
        for member in members_in_fold:
            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)
            values[[position[study] for study in member['ids']]] = np.asarray(member[pred_key], np.float64)
            if np.isnan(values).any():
                raise WeightsError(f"{member.get('id')}: incomplete {pred_key} coverage")
            matrices.append(values)
        raw_fold_mean = np.mean(matrices, axis=0)
        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method='average', pct=True).to_numpy(np.float64))
        diagnostics.append({'ensemble_group': key, 'members': len(members_in_fold)})
    if len(fold_ranks) != 5:
        raise WeightsError(f'legacy branch requires five folds, found {len(fold_ranks)}')
    return (all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics))

def blend_legacy_frontier_and_soft(frontier_rank, soft_rank):
    output = np.asarray(frontier_rank, np.float64).copy()
    for j, target in enumerate(TARGETS):
        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))
        if alpha:
            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]
    return output

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):
        if float(np.std(pred)) < 1e-09:
            log(f"  {m['id']}: degenerate predictions; not banked")
            return
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': public_pred, 'soft_pred': public_soft})
            elif public_pred is not None:
                log(f"  {m['id']}: public-frontier vote omitted because only {len(starts)} / {len(starts_full)} windows completed")
            all_ids, acc = _combine(per_member)
            write_submission(acc, all_ids, test_df, 'submission.csv')
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s){(', jitter' if jitter else '')}); submission.csv = weighted rank mean of {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, 'test '), f'test g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                left = TIME_BUDGET - (time.time() - T0)
                remaining = len(pending) + left_after
                slots_left = -(-remaining // len(DEVS))
                starts, jit = (starts_full, False)
                if est['fixed'] is not None and est['win'] is not None:
                    afford = max(left * 0.9, 0.0)
                    room = afford / max(slots_left, 1)
                    if est['fixed'] + est['win'] > room:
                        log(f'  {left / 60:.0f} min left: surrendering {len(pending)} member(s); not one more fits')
                        pending.clear()
                        return (None, None, False)
                    jit = est['fixed'] + 2 * len(starts_full) * est['win'] <= room * 0.6
                    per_win = est['win'] * (2 if jit else 1)
                    n_win = int((room - est['fixed']) / per_win) if per_win > 0 else len(starts_full)
                    n_win = max(1, min(len(starts_full), n_win))
                    if n_win < len(starts_full):
                        mid = (len(starts_full) - n_win) // 2
                        starts = starts_full[mid:mid + n_win]
                return (pending.pop(0), starts, jit)

        def worker(dev):
            others = [d for d in DEVS if d is not dev]
            while True:
                m, starts, jit = pop_next()
                if m is None:
                    return
                for attempt, d in enumerate([dev] + others[:1]):
                    try:
                        p, public_p, public_soft, (fs, ws) = _run_member(path, m, d, Cte, Mte, idx, starts, jit)
                        with STATE_LOCK:
                            est['fixed'], est['win'] = (fs, ws)
                        bank(m, st_te, p, starts, jit, public_p, public_soft)
                        break
                    except Exception as exc:
                        log(f"  MEMBER {m['id']} failed on {d} ({type(exc).__name__}: {exc}); " + ('retrying on peer device' if attempt == 0 and others else 'dropped -- costs one vote, not the run'))
                        if d.type == 'cuda':
                            with torch.cuda.device(d):
                                torch.cuda.empty_cache()
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        del Cte, Mte
        gc.collect()
    if not per_member:
        raise WeightsError('no member produced predictions; submission stays at 0.5')
    all_ids, acc = _combine(per_member)
    sub = write_submission(acc, all_ids, test_df, 'submission.csv')
    log(f'final submission.csv = weighted rank mean of {len(per_member)} member(s); {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    if len(public_frontier_members) == len(members):
        frontier_ids, frontier_acc = _combine(public_frontier_members)
        frontier_sub = write_submission(frontier_acc, frontier_ids, test_df, 'submission_public_0899.csv')
        log(f'submission_public_0899.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {frontier_sub.shape}; nulls {int(frontier_sub[TARGETS].isna().sum().sum())}')
        fold_ids, fold_frontier, fold_diagnostics = combine_public_members_by_fold(public_frontier_members, 'pred')
        soft_ids, fold_soft, _ = combine_public_members_by_fold(public_frontier_members, 'soft_pred')
        if fold_ids != soft_ids:
            raise WeightsError('legacy hard/soft study order mismatch')
        legacy_prediction = blend_legacy_frontier_and_soft(fold_frontier, fold_soft)
        legacy_sub = write_submission(legacy_prediction, fold_ids, test_df, 'submission_legacy_fold_blend.csv')
        fold_diagnostics.to_csv('legacy_fold_diagnostics.csv', index=False)
        log(f'legacy DINO aggregation written from five folds; {legacy_sub.shape}')
    else:
        log(f'public-frontier fallback not emitted: {len(public_frontier_members)} / {len(members)} required public members completed')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")

def augment(imgs, generator=None):
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)

def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def find_dinov2(variant='small'):
    if not (DINO / 'config.json').is_file():
        raise FileNotFoundError(DINO)
    return DINO

def legacy_group_members():
    return {}

def run_dinov2():
    path = ASSET / 'rsna-knee-weights'
    infer_from_package(path, DEVS[0])
    public = Path('/kaggle/working/submission_public_0899.csv')
    if not public.is_file():
        raise RuntimeError('public DINOv2 frontier was not produced')
    public.replace('/kaggle/working/submission.csv')
    for name in ('submission_legacy_fold_blend.csv', 'legacy_fold_diagnostics.csv'):
        candidate = Path('/kaggle/working') / name
        if candidate.is_file():
            candidate.unlink()
run_dinov2()


[    0.2s] memory: 29.9 GB available, 13.5 GB to the cache; sizing for 4407 train + 1322 test studies -> 1 group(s) of 3 = 3 slices per slot
[    0.2s] weights package: 20 member(s) from /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-weights; 2 device(s)
[    0.3s] test header pass: 15 series
[    0.3s] decode group 1/1: 336px x 12 slices, crop 130.0 mm -> 20 member(s)
[    0.3s] test laterality: 1 from the tag, 2 from geometry, 0 unresolved; tag and geometry disagree on 0 (0.0% of the tagged)
[    0.3s] test g1: cache (3, 6, 12, 336, 336) = 0.0 GB
[    0.3s] test g1: ordering 12 slot-series (469 slice headers)
[    2.3s] test g1: ordered 12/12 by geometry (0 kept arbitrary) in 2s
[    2.3s] test g1: decoding 12 slot-series
[    3.1s]   test g1 12/12
[    3.2s] test g1: 12/12 slots filled


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   25.7s] backbone: 12 blocks, last 6 trainable (10.7M params), feature dim 768
[   27.6s] e5427d6c21: fingerprint matches within 2.8e-06


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   28.6s] backbone: 12 blocks, last 6 trainable (10.7M params), feature dim 768
[   29.1s]   banked e5427d6c21 fold 2 (10 window(s)); submission.csv = weighted rank mean of 1 member(s)
[   29.5s] d47313baef: fingerprint matches within 2.1e-06
[   30.4s]   banked d47313baef fold 2 (10 window(s)); submission.csv = weighted rank mean of 2 member(s)


...

[   54.4s] backbone: 12 blocks, last 6 trainable (10.7M params), feature dim 768
[   54.7s] 1e553ba481: fingerprint matches within 1.7e-06
[   55.1s]   banked 7f8321affc fold 3 (10 window(s), jitter); submission.csv = weighted rank mean of 17 member(s)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   55.9s] backbone: 12 blocks, last 6 trainable (10.7M params), feature dim 768
[   56.5s] b7c37cc80e: fingerprint matches within 1.1e-06
[   56.5s]   banked 1e553ba481 fold 1 (10 window(s), jitter); submission.csv = weighted rank mean of 18 member(s)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[   57.4s] backbone: 12 blocks, last 6 trainable (10.7M params), feature dim 768
[   57.7s] f335c812fe: fingerprint matches within 2.9e-06
[   58.2s]   banked b7c37cc80e fold 3 (10 window(s), jitter); submission.csv = weighted rank mean of 19 member(s)
[   59.4s]   banked f335c812fe fold 1 (10 window(s), jitter); submission.csv = weighted rank mean of 20 member(s)
[   59.6s] final submission.csv = weighted rank mean of 20 member(s); (3, 13); nulls 0
[   59.6s] submission_public_0899.csv = exact no-jitter public-frontier rank mean of 20 member(s); (3, 13); nulls 0
[   59.6s] legacy DINO aggregation written from five folds; (3, 13)


In [4]:
_A5_SAVED = dict(globals())
import gc, os, time, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
COMP = Path(_COMP_ROOT)
CKPT = ASSET / 'knee-mri-fold-weights'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'competition : {COMP}')
print(f'checkpoints : {CKPT}')
print(f'device      : {DEV}')
for i in range(torch.cuda.device_count() if DEV == 'cuda' else 0):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, {torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB, native bf16={cc >= (8, 0)}')
SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.exists():
    SERIES_ROOT = COMP / 'train_series'
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx, study, recs = args
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
            if not files:
                continue
            flip = plane != 'Sagittal' and series_side(files[0]) < 0
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            if INTENSITY == 'series':
                crops = [read_crop(files[p]) for p in picks]
                got = [x for x in crops if x is not None]
                if got:
                    samp = np.concatenate([x[::4, ::4].ravel() for x in got])
                    lo_, hi_ = np.percentile(samp, [1, 99])
                    for c, x in enumerate(crops):
                        if x is None:
                            x = read_crop(files[min(len(files) - 1, picks[c] + 1)])
                        if x is not None:
                            out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)
            else:
                for c, p in enumerate(picks):
                    img = render(files[p], flip)
                    if img is None:
                        img = render(files[min(len(files) - 1, p + 1)], flip)
                    if img is not None:
                        out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return (idx, out, mask)
sub_df = pd.read_csv(COMP / 'sample_submission.csv')
ser_csv = pd.read_csv(COMP / 'test_series.csv')
if not (COMP / 'test_series').exists():
    ser_csv = pd.read_csv(COMP / 'train_series.csv')
ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
studies = sub_df.StudyInstanceUID.tolist()
by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
print(f'{len(studies):,} test studies, {len(by):,} with series metadata')
N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models = []
for ckpt_path in sorted(CKPT.glob('*_f*.pt')):
    z = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = z['cfg']
    _stem = cfg.get('stem', 'native')
    _in = 3 if _stem == 'compress' else cfg.get('n_slice', 16)
    enc = timm.create_model(cfg['backbone'], pretrained=False, num_classes=0, in_chans=_in, **{'img_size': cfg['img']} if 'vit_' in cfg['backbone'] else {})
    if cfg['cond'] == 'token':
        enc = ViTSlotToken(enc, N_SLOT_TYPES)
    m = Net(enc, cfg['cond'], cfg.get('n_meta', 0), cfg['pool'], stem=_stem, n_slice=cfg.get('n_slice', 16))
    missing, unexpected = m.load_state_dict(z['state_dict'], strict=False)
    assert not missing, f'missing {missing[:5]}'
    assert not unexpected, f'unexpected {unexpected[:5]}'
    models.append(m.eval())
    print(f"loaded {ckpt_path.name}  fold {z['fold']}  {cfg['backbone']} pool={cfg['pool']} meta={cfg['meta']}")
CFG = cfg
assert CFG.get('n_meta', 0) == 0, f"checkpoint expects {CFG['n_meta']} metadata features -- build slot_meta for the TEST studies and pass it to predict() before submitting"
print(f"\n{len(models)} fold models ready | input norm: {CFG.get('norm', 'none')}")
AMP_PREF = 'bf16'

def amp_for(dev):
    if not str(dev).startswith('cuda'):
        return (torch.float32, False)
    cc = torch.cuda.get_device_capability(dev)
    if AMP_PREF == 'bf16':
        return (torch.bfloat16, True)
    if AMP_PREF == 'fp16':
        return (torch.float16, True)
    if AMP_PREF == 'fp32':
        return (torch.float32, False)
    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
AMP_DT, AMP_ON = amp_for(DEV)
WORKERS = max(1, min(4, os.cpu_count() or 4))
CHUNK = 48
MICRO = 8
models = [m.to(DEV).eval() for m in models]
print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

def _norm_(im):
    k = CFG.get('norm', 'none')
    if k == 'zscore':
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-06) * m
    if k == 'imagenet':
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _micro(images, masks):
    dev = DEV
    ims, slots, sidx, vms = ([], [], [], [])
    for b in range(len(masks)):
        present = np.nonzero(masks[b] > 0)[0]
        if len(present) == 0:
            continue
        blk = images[b][present]
        ims.append(torch.from_numpy(blk))
        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
        slots.append(torch.from_numpy(present + 1).long())
        sidx.append(torch.full((len(present),), b, dtype=torch.long))
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    if not ims:
        return out
    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
    sl = torch.cat(slots).to(dev)
    si = torch.cat(sidx).to(dev)
    vm = torch.cat(vms).to(dev)
    sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
    per = torch.zeros(len(models), len(masks), len(LABELS), device=dev, dtype=torch.float32)
    with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu', dtype=AMP_DT, enabled=AMP_ON):
        for fold_index, model in enumerate(models):
            per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(masks), vm=vm).float())
    got = per.cpu().numpy()
    keep = np.array([(masks[b] > 0).any() for b in range(len(masks))])
    out[:, keep] = got[:, keep]
    return out

def predict(images, masks):
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    for a in range(0, len(masks), MICRO):
        b = min(a + MICRO, len(masks))
        out[:, a:b] = _micro(images[a:b], masks[a:b])
    return out
preds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)
t0, done = (time.time(), 0)
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for c0 in range(0, len(studies), CHUNK):
        block = studies[c0:c0 + CHUNK]
        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
        msks = np.zeros((len(block), N_SLOT), np.uint8)
        futs = [ex.submit(build_study, (i, s, by.get(s, []))) for i, s in enumerate(block)]
        for f in as_completed(futs):
            try:
                i, a, k = f.result()
                imgs[i], msks[i] = (a, k)
            except Exception as e:
                print(f'  study failed: {type(e).__name__}: {e}')
        preds[:, c0:c0 + len(block)] = predict(imgs, msks)
        done += len(block)
        el = time.time() - t0
        print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
        del imgs, msks
        gc.collect()
print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
A5_W = 0.45
A5_LABELS = list(LABELS)
_a5_ok = np.isfinite(preds).all(axis=(0, 2))
_a5_rank_mean = np.zeros((len(studies), len(LABELS)), np.float64)
for fold_index in range(preds.shape[0]):
    fold = preds[fold_index][_a5_ok]
    ordinal = fold.argsort(0).argsort(0).astype(np.float64)
    _a5_rank_mean[_a5_ok] += ordinal / max(len(fold) - 1, 1)
_a5_rank_mean /= preds.shape[0]
_a5_rank_mean[~_a5_ok] = np.nan
A5_PREDS = dict(zip(sub_df['StudyInstanceUID'].astype(str), _a5_rank_mean.astype(np.float32)))
for _a5k, _a5v in _A5_SAVED.items():
    globals()[_a5k] = _a5v
del _A5_SAVED, _a5k, _a5v
_a5_sub = pd.read_csv('/kaggle/working/submission.csv', dtype={'StudyInstanceUID': str})
assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'
if A5_W > 0:
    _a5_ours = np.stack([A5_PREDS[_u] for _u in _a5_sub['StudyInstanceUID'].astype(str)])
    _a5_base_rank = _a5_sub[A5_LABELS].rank(method='average', pct=True)
    _a5_ours_rank = pd.DataFrame(_a5_ours, columns=A5_LABELS, index=_a5_sub.index).rank(method='average', pct=True)
    _a5_sub[A5_LABELS] = (1.0 - A5_W) * _a5_base_rank + A5_W * _a5_ours_rank
    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()
    _a5_sub.to_csv('/kaggle/working/submission.csv', index=False)


competition : /kaggle/input/competitions/rsna-knee-abnormality-detection
checkpoints : /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/knee-mri-fold-weights
device      : cuda
  gpu0       : Tesla T4 sm_75, 15 GiB, native bf16=False
  gpu1       : Tesla T4 sm_75, 15 GiB, native bf16=False
series root: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
3 test studies, 3 with series metadata
loaded m_f0.pt  fold 0  vit_small_patch16_dinov3.lvd1689m pool=xcodex meta=none
loaded m_f1.pt  fold 1  vit_small_patch16_dinov3.lvd1689m pool=xcodex meta=none
loaded m_f2.pt  fold 2  vit_small_patch16_dinov3.lvd1689m pool=xcodex meta=none
loaded m_f3.pt  fold 3  vit_small_patch16_dinov3.lvd1689m pool=xcodex meta=none
loaded m_f4.pt  fold 4  vit_small_patch16_dinov3.lvd1689m pool=xcodex meta=none

5 fold models ready | input norm: none
device cuda | amp bfloat16 (on=True) | workers 4 | chunk 48 | micro 8
  3/3  0.0m  eta 0.0m

inference done in 0.1 min


In [5]:
from __future__ import annotations
import contextlib as _rad_contextlib
import base64 as _rad_b64
import zlib as _rad_zlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath
import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50
_RAD_LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_RAD_ALPHA = 0.5
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_REFERENCE_HEADS_SHA256 = '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa'
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_E13_HEADS_SHA256 = 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a'
_RAD_E13_MEMBER_WEIGHT = 0.5
_RAD_V48_SECOND_ALPHA = 0.15
_RAD_CAL_PAYLOAD = 'eNrtmk1vI8cRhv9KsJdcKKE/q6tzc4z4ZCMBcjQWhrCRDSG2ZEjaIEGQ/57n7RlRQ3KG4jqLJAcDS4o709NdXR9vvVU9/3z30+3N/bvffRuuawgxlm7eq2ePeffrpV8v/V9euvLraD3k6ilW6zn126vYd+U6eKmx9RiaF7OSx+X1weE67K7SdSgpVSauuaecUxr3rtp1LK0HyyV7y9Gmy/E6pBhTL61Zt2gWx+WtSew6JmOoM5AHutXp+ro8+ZrlsnvJOfDdfRL+Kl4zd2bqllNmga7rvrva2OzGNFuynGxpmnxrS/069hCtpt6T1dLLOQVsTbJ1fWunG2qXAX8NiU+7VK8rdqvNU89uwQwjpWyeLdTCnVy84ua9pthSjznHXLjDpZxbLe4WPRAQCT+LrSVcGGdLzNX0XKhesC2eV5uZ67lQPDlmSzUhS2+61ELtoTOyWGjdPudc73fvnj7c/Hg7ElpyzYXjdwIZ32m7X3INZ0crtvtc833ua6fyoUYUGf4H13rJpX+GK5aa5VbeWO9ye/03bLi97i8SpaSCl49nc4gdxI2p1dZyVmQnfL56jBH/j70GqSq6dyMROLHQGvCpcSnkYsyeohNErnVjbamVjs47wOpBa7BAsa61SXulDfSIwAbAkQqDmSK38WwImKK1kFJ3d11CpFqR1mPHlj1pWVAHeDfyU2hk0vGogz0GqCBbKmFMx9xIWEAbQ8I4PUvmAplQCeuij7GNXGMk3yhBsFIfEnvVE6HFYMHDtFuLzKwpya9gisaZduAkcyAvmw1ROjmd7OnBYytpOBqJjTyK4KzT0Pi0tQJYslc0nGqcNZV7dEaRvfcSewg9h4p41iwO+4CWMkNG1326VBmHFUoB7VoakzE+JAtYN/NknS9lLJ+9S5qR56Q2kLA4qgTuplGNmUJAkbXiGbrU0HCIsgtYaGOUVXbayNeykLfpGv8lulBfgiFEmx61xm3LTlrOPoa51Ng7IyJiT4tWxrE0ZmnJRnhawZsyLgjXIC9MRu0tRAgImkJ7bUyHKk1eUgIewtRjXGDJqO1mNJrHfNUty1HRW2SSoV3FAVIA//hrUXKANxhbRbEkgqAFKiCC43qOEXsPTaKtKCdkqx3/koMYsuP+mh4HrHoQtqSIw/h4DM7EpRL5zRirAcHiUDj2SUlr9fFAsElHBX/zWgJCQlw+83Qksw8Pt9+Ty0hmOBQBh9XQAr7fxjRQ2IkGTT/w8cAiBKwRc1jwcMh+WKtMgFShNaDGJWRAechNSOMUldXTynNIUPAaEKKkCv1cLFzxaowBOmX8vU8Pj1voWa5JTPFcGN4QXm+PpZPjA8QQYYp9Qab9rZiIAuLX8AA8f/jX4kHgI+BXCtBEeOTFvNPapIyC92YAEOmHypJcCSiFOpQi712M73IL8KiBX8Hz62pDNsAH8MK/rMR+tNSRIRJwAkIIY8Rn/fD2kB2V9I7BMX+KSJ/XJtNAbwFglmQZbu/90CZcMhAeZKudKAvlSBLwNwP1aA88BYpPdJRkbADGeBzWN2IOvySVELxyU0Lx0JHQHsFpEQRsCB9iO1qT3IOHGUjMDqeczX4BiIbvCjkiLD6t6G239p+gAML1KQuAdaLLdidDV6Y6uPR+9+3LT8KrlYgzAsjg7ok+VJakimvgAjn5qUFmtTGJKXGxSadg2UuLkWok4EvGJ0Hdsr+DmpVlGsUMZkxtuTI8vDAVaIsnSIDbK0DhVSpAFlu4iRtgrLYWRaArADOcFDAfErEdwBS8dWpNqHupW3o7nIukTmxlmASq4L/51ESrznqJTSkgCzslVSMncekb8659ljeyYttxK0oX9k50n3h2kDqoapToWt8S9/yO3tTXyteLu+19rkPViKkIblLnzJhJUhYENUJCtdfWKkEFe9WTsWin6azpqJFIfEwNyLNes+PZXFV3h+tIOXjb5mzIRnCLaWKuhnPtjsCVPAOwdkhQhBSMfWLVruTgwEM/WXt1FYgv5AN4IsyBiHrOSscCBjIomksgZCPFn9grqDrE9UXDQCSPfs5RXyeuQl3gwcEIdn5JzADhpG34s4uF5NQ24eh1GWISkEmYA6iFiNezYAbkKNGJhIk+l9XNXK3r7AgLT4YnRUuHicJS0NRLLG0KDrF1IbkaMzj2MoeKzAH/kTo9KsnWJe8gDZUuxs1iPi0CayABMdLIT4qQCbfENfCiAsujIsj9KMUQ1kXwXUBFZspHZm9Zj/dB/SrVy1qOJg9ApKAlmSQNpr6SjibnNgVoqZQCr8gllgsTiFAOCFSeDcYWuKOChZQXi5dPxouF5Oo6ogVTaxDwnVE8KfQFI6Zomwgb/oI4VG2Ae1MdsTQCwQuZp5ZROZPcdudWHrZRfwe4cPE1cv9L/FDuwRwGNYOttGllMRItS1K3sFDQDAwQMvgpfIzyIeQj2yTFOhomGLsBVPtAASGLqiXKnChASW/4ILUTGlftQlUVl8FDjifbKYIzLKXmco4anNPK6ZVl8MhVBs+D5YgTo8HXuIEDQDJF4wHEYL5PvEExn4PIt7x0JumDeUgjxHeFIZDt+6xN5mALCc6pMjTZ7oy8EomCUA09MTRvvuCLxyFC+sEWCGiqjE+HIAJ4g39Bi5sadSN9MJZaFWLbpubBNBvlmwrPAONJWS1DpaIgbyKuZSbKg7qZzgNKsqnwIX8RAOvINdgfRg0jPNJBWE+5xAQ9qDaAPh4nKInagjAL9otj26U+sAnbUERTF0QYjHV8j8OEA+mohtH9SLLuoUIh6uAWHI6yAw54uEsqL6zufMAAt3yW+6xSFLmKwEmYgIPlXHYbPAo3AvKovCk/0KR5m3fmKkpFz2W3ng58h7KzqVVHqVYUpfGlAJHvQ2dN7QKmXoQITggHUz9HGZhSty5smYHeSL4pFdvbXlXZcSJV7GCHKPpsePEmVAOm41WxbuW3Y+qkrhYhisbrSBqbGiGdAkBwd5gzNehxGUVhzu5JCKa2VAyL2pfYgYgUtcTwu3RKZzO2JublhfCDbO1Qqypu8SdTWO0501Z6iAAH68g21mECvjX8bZZ7yvpViqq9BlHCnhj7DFuiXm8qEwgrAkK9hpfbuU/nN/i4l7I/qQEnpR4qVHUxgDjblWuM2UZFhOpzp+apb8i4ua8gnyHrqPMHs839gK7gaaYDDtILqNTWOF8kxVYdMxlVDwE68/F8DR2CgpCSkkzkEvKY3x9GoRqjpn8ZLj62TyEoRVV1dxrV2GtNOMosPAjqSC6pm6yRKRJFM5wnixbhh6uLl9FdjElsUvX7WxR61T1gGohOkQaej27MxiR+DQjAndQ5SgL4Yb+VMKT2UncGK4se5hfeR7Jr6nygbTJcPGK7QSpXOoBJkuMXlFTYQX0aRsObYEQn22BNrDQdKwmZ1DDaZNcANxumdkGs3pZIJcaCcxBuPSlGziTf+UNZIgqmLMTWS1/XYNBxFowH5FchZb7uT5EQqervUK+Rc15pP55mBpKra6Wju5MCbZwiKOSS/HFuZyVRBFeeIoVrc35oNnVt1ARpKDaHF2BO1z3DLILOrOVZdhIG52ojyFxsVa1k6Bq+sOS7AA0upIqpiCt9Om8+1SsIoI4/ZYFKZq9tdwnhCzrVRpoCxpqSo+0ua1DNLCOMc3X19vGSZZ9znfEAMTUpymSGqYW/kpXU6FWV6+pah9OGLveaTkopOV1d3U9oWfAhMiyK46fRgPeL9LTSDAtUjj5cFN4D8S5zmaCjmaTjJ/Kvxb3MaFrHVI27QS2vRfcMCmeqcUXY+NX6652okyj19OEC3SaFrdWyeyJMxo7gPrANn17GjQ5RgtqvregNjzr1hYOOxATeos4b/IItGRxEFZC4aNruVL1ZbGx8ojpE5moLiGNavazB+baxPoXr/gK56zjI1VGbjtG8nA3Xg3SpFl2gtqzqsM/oGgb5Axj4w+XD1g48MBFrMAnR+TRxMci1+8h+uCC1e1nmC7XEWhf2zEdIw5SsCe3Tcc18XibsU/PKhJhd7a380s67RLEvQQWUw9KKjmps7qefVeub/IZQoLbK0qxnHRFP7qlOy8BURC7Fy2LDPk7N1F1VEm1lrZbSmSWVSoNk63DQXgvIohAjYcjXU7b/zI8u7RVvPiRChSqVjkiU6YQjrT1ImVAAN/WoKEp62V2aQGAdAuQKhSW5qouyIdQ4jNe5NS6I8v10hLIeqIIMqplDn71o0dYl4fEFkRZ4LmhFJNUSQqZCNlBT2fIWnKzJ9XWwG48bOzGZWEIffcVx/q23xAziBV83gf1cE4/+mvI8/EFV3QVmPW0a6rB7ZD314ploTllOSlFYqX3X4u7TJg6jmxLRWxit1FaPtArKoHfHeb3jwPmNGDqfvS+Iv9Ns14Vv6p9rfyft+HGiFqmJIQSU++rlbegPBqDumli9zoOGSptec8O6GAWqtaAgkOgqZ3P1WhQQ08aj2KGONuEIqdZetzuLFB6qQniid8HkZTnlyGsPGnA4lY5Qu1456WnbokG9IupwU0NoPhHTwRRUqynZU0HObV8QUyZuo/lb6tFhsdxb7bCoU9qWe8vLxBwj2laVzgS+bIZGMfee1Qaldl87gBZ5jjr3Y4rq2Xaf3LatohHwxzbqBKtv4d8bHnh1eUqfVRx07jc4zfzySrj8IGV9NMFX4mBKrq6FXc4Jz154/3737u7++fbxw+3Pz9N7eg0X0mHCeHfHbHrNhrCIpdXRA/LRRC4WR9sU/ZqJB46XJsJouwU1rPT+il6uKHqNAdbJ2InUJp0wVWBV7w8NuldU7uMyajZqh2N6UfeoM6ym1zLGizF6T6AnNQZiHO/KZMijZMOV2/yKQFKv0TSXq0Hb5teE9F4HLp50QKJ3OX64edZ7ie+++PLrd7t339z+5e7mx9/88Qt+f82dx5f//Omr6e8fvv/+49Pdwz0/f3/z19vH3z7x68uH++fpKhP+/Pjw/PDh4cfv+Hz86f5Jk99/93T7eHersfff/fnmh/H3y4fH8feLv9/x96ub5+l7vq9f0wj9msf8+HH6fhnDr3kMvzRGG3p8+PizVt3vaXy/ysjox5sPzx8fbxn+7cuWv7m9v3v68PFpsfH9pcWwLc1oyEI3f/7H/cPf7p7vnhZ6ev/+X/8GYIe3xg=='
_RAD_CAL_W = 0.40
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = (2048, 512)
_RAD_E11_SLOTS = [('SAG_NOFS', 'Sagittal', None, False), ('COR_NOFS', 'Coronal', None, False), ('AX_NOFS', 'Axial', None, False), ('SAG_FS', 'Sagittal', None, True)]
_RAD_E11_CROP_MM = 130.0
_RAD_E13_SLOTS = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True), ('SAG_NOFS', 'Sagittal', None, False)]
_RAD_E13_CROP_MM = 130.0
_RAD_E13_CACHE_SLICES = 8
_RAD_E13_IMG = 224
SLOTS = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True)]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8

def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()

def _rad_find_file(name, expected_sha=None, explicit_env=None):
    files = {_RAD_ENCODER_SHA256: ASSET / 'resnet-50-radimagenet-marwan/ResNet50.pt', _RAD_REFERENCE_HEADS_SHA256: ASSET / 'rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt', _RAD_E13_HEADS_SHA256: ASSET / 'kernel-sources/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt'}
    candidates = []
    bundled = files.get(expected_sha)
    if bundled is not None and bundled.is_file():
        candidates.append(bundled)
    candidates.extend(find_input_files(name, preferred=('repro-assets', 'radimagenet', 'e11')))
    checked = []
    seen = set()
    for path in candidates:
        path = Path(path)
        key = str(path)
        if key in seen or not path.is_file():
            continue
        seen.add(key)
        checked.append(key)
        if expected_sha is None or _rad_sha256(path) == expected_sha:
            print(f'RadImageNet asset: {path}', flush=True)
            return path
    raise FileNotFoundError(f'{name} with expected hash was not found; checked={checked}')

class _RadEncoder(_rad_nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(*list(_rad_resnet50(weights=None).children())[:-2])

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))

class _RadHead(_rad_nn.Module):

    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(_rad_nn.LayerNorm(_RAD_TOKEN_DIM), _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM), _rad_nn.GELU())
        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * 0.01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * 0.01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.attn = _rad_nn.MultiheadAttention(_RAD_HEAD_DIM, 8, dropout=0.1, batch_first=True)
        self.fuse = _rad_nn.Sequential(_rad_nn.LayerNorm(_RAD_HEAD_DIM * 4), _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM), _rad_nn.GELU(), _rad_nn.Dropout(0.15))
        self.weight = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), N_SLOT, CACHE_SLICES, _RAD_HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token, key_padding_mask=key_padding, need_weights=False)[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat([attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias

def _rad_load_public_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_radimagenet_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=True)
    expected = {'version': 'v52-radimagenet-resnet50-official-1', 'targets': _RAD_LABELS, 'encoder_sha256': _RAD_ENCODER_SHA256, 'encoder_source_commit': '0ce16f7375db4236e646829d1eca61cdb4282133', 'img': 224, 'slices_per_plane': 8, 'feature': 'global_average_pool'}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'public-v15 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('public-v15 bundle requires exactly five heads')
    if sorted((int(record.get('fold', -1)) for record in folds)) != list(range(5)):
        raise RuntimeError('public-v15 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return (heads, str(heads_path))

def _rad_load_e13_heads(device):
    heads_path = _rad_find_file('v52_e11_heads.pt', _RAD_E13_HEADS_SHA256)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {'version': 'e11-radimagenet-resnet50-diverse-1', 'targets': _RAD_LABELS, 'encoder_sha256': _RAD_ENCODER_SHA256, 'slots': [list(slot) for slot in _RAD_E13_SLOTS], 'crop_mm': _RAD_E13_CROP_MM, 'img': _RAD_E13_IMG, 'slices_per_plane': _RAD_E13_CACHE_SLICES, 'feature': 'global_average_pool'}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E13 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E13 bundle requires exactly five heads')
    if sorted((int(record.get('fold', -1)) for record in folds)) != list(range(5)):
        raise RuntimeError('E13 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return (heads, str(heads_path))

@_rad_torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask, device):
    n, slots, slices, height, width = pixels.shape
    features = _rad_np.zeros((n, slots * slices, _RAD_TOKEN_DIM), _rad_np.float16)
    token_mask = _rad_np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = _rad_np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, height, width)
    batch = 192 if device.type == 'cuda' and _rad_torch.cuda.device_count() > 1 else 96 if device.type == 'cuda' else 8
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = _rad_torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = _rad_torch.autocast('cuda') if device.type == 'cuda' else _rad_contextlib.nullcontext()
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not _rad_np.isfinite(values).all():
            raise RuntimeError('V36 non-finite RadImageNet feature')
        features.reshape(-1, _RAD_TOKEN_DIM)[indices] = values.astype(_rad_np.float16)
    return (features, token_mask.astype(_rad_np.float32))

@_rad_torch.inference_mode()
def _rad_predict_head(head, features, masks, device, batch=64):
    predictions = []
    for start in range(0, len(features), batch):
        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)
        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)
        amp = _rad_torch.autocast('cuda') if device.type == 'cuda' else _rad_contextlib.nullcontext()
        with amp:
            predictions.append(_rad_torch.sigmoid(head(image, mask)).float().cpu())
    return _rad_torch.cat(predictions).numpy()

def _rad_rank_columns(values):
    return _rad_pd.DataFrame(_rad_np.asarray(values, dtype=_rad_np.float64)).rank(method='average', pct=True).to_numpy(_rad_np.float64)

def _rad_validate(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('V36 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('V36 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError('V36 invalid submission values')


def _v18_cal_protocol(uids):
    frame = _rad_pd.read_csv(
        ROOT / 'test_series.csv',
        dtype={
            'StudyInstanceUID': str,
            'SeriesInstanceUID': str,
        },
    )

    frame['StudyInstanceUID'] = (
        frame['StudyInstanceUID'].astype(str)
    )

    index = _rad_pd.Index(
        [str(uid) for uid in uids],
        name='StudyInstanceUID',
    )

    table = _rad_pd.DataFrame(index=index)

    table['n_series'] = (
        frame.groupby('StudyInstanceUID')
        .size()
        .reindex(index)
        .fillna(0)
    )

    for plane in (
        'Sagittal',
        'Coronal',
        'Axial',
    ):
        part = frame[
            frame['Anatomical_Plane']
            .astype(str)
            .eq(plane)
        ]

        table[f'n_{plane[:3]}'] = (
            part.groupby('StudyInstanceUID')
            .size()
            .reindex(index)
            .fillna(0)
        )

    for flag in (
        'Fat_Suppression',
        'Fluid_Sensitive',
    ):
        marked = frame[
            _rad_pd.to_numeric(
                frame[flag],
                errors='coerce',
            )
            .fillna(0)
            > 0
        ]

        prefix = flag[:3]

        table[prefix] = (
            marked.groupby('StudyInstanceUID')
            .size()
            .reindex(index)
            .fillna(0)
        )

        for plane in (
            'Sagittal',
            'Coronal',
            'Axial',
        ):
            part = marked[
                marked['Anatomical_Plane']
                .astype(str)
                .eq(plane)
            ]

            table[f'{prefix}_{plane[:3]}'] = (
                part.groupby('StudyInstanceUID')
                .size()
                .reindex(index)
                .fillna(0)
            )

    return table


def _v18_calibrate_transformer(
    branch,
    baseline_rank,
    public_rank,
    pass2_rank,
    expected_ids,
):
    payload = _rad_json.loads(
        _rad_zlib.decompress(
            _rad_b64.b64decode(
                _RAD_CAL_PAYLOAD
            )
        ).decode()
    )

    gate = set(payload['gate'])

    protocol = _v18_cal_protocol(
        expected_ids
    )

    if (
        protocol.columns.tolist()
        != list(
            payload[
                'protocol_columns'
            ]
        )
    ):
        raise RuntimeError(
            'V18 calibration protocol '
            'layout mismatch'
        )

    mean_rank = (
        baseline_rank
        + public_rank
        + pass2_rank
    ) / 3.0

    blocks = [
        baseline_rank,
        public_rank,
        pass2_rank,
        public_rank - baseline_rank,
        pass2_rank - baseline_rank,
        mean_rank,
    ]

    for group in payload['groups']:
        columns = [
            _RAD_LABELS.index(target)
            for target in group
        ]

        blocks.append(
            mean_rank[
                :,
                columns,
            ].mean(
                axis=1,
                keepdims=True,
            )
        )

    blocks.append(
        protocol.to_numpy(
            _rad_np.float64
        )
    )

    x = _rad_np.concatenate(
        blocks,
        axis=1,
    )

    centre = _rad_np.asarray(
        payload['mean'],
        _rad_np.float64,
    )
    spread = _rad_np.asarray(
        payload['scale'],
        _rad_np.float64,
    )
    coef = _rad_np.asarray(
        payload['coef'],
        _rad_np.float64,
    )
    bias = _rad_np.asarray(
        payload['intercept'],
        _rad_np.float64,
    )

    if (
        x.shape[1] != 88
        or coef.shape != (
            len(_RAD_LABELS),
            88,
        )
    ):
        raise RuntimeError(
            f'V18 calibration feature drift: '
            f'x={x.shape}, coef={coef.shape}'
        )

    spread = _rad_np.where(
        _rad_np.abs(spread) > 1e-8,
        spread,
        1.0,
    )

    adjusted = _rad_rank_columns(
        (
            (
                x - centre
            )
            / spread
        )
        @ coef.T
        + bias
    )

    output = branch.copy()

    values = output[
        _RAD_LABELS
    ].to_numpy(
        _rad_np.float64
    ).copy()

    for index, target in enumerate(
        _RAD_LABELS
    ):
        if target in gate:
            values[
                :,
                index,
            ] = (
                (
                    1.0
                    - _RAD_CAL_W
                )
                * values[
                    :,
                    index,
                ]
                + _RAD_CAL_W
                * adjusted[
                    :,
                    index,
                ]
            )

    output[
        _RAD_LABELS
    ] = _rad_rank_columns(
        values
    )

    _rad_validate(
        output,
        expected_ids,
    )

    return output, gate



def _rad_main():
    work = _RadPath('/kaggle/working')
    primary = work / 'submission.csv'
    test = _rad_pd.read_csv(ROOT / 'test.csv', dtype={'StudyInstanceUID': str})
    expected_ids = test.StudyInstanceUID.astype(str).tolist()
    baseline = _rad_pd.read_csv(primary, dtype={'StudyInstanceUID': str})
    _rad_validate(baseline, expected_ids)
    device = _rad_torch.device('cuda:0')
    test_series = _rad_pd.read_csv(ROOT / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))

    def cache(slots, crop, tag, threshold):
        globals().update(SLOTS=list(slots), N_SLOT=len(slots), CACHE_SLICES=8, IMG=224, CACHE_IMG=224, CROP_MM=float(crop), RULES=dict(RULES_LEGACY))
        headers = annotate(walk('test_series'))
        studies, pixels, masks = build_cache(pick_slots(headers, plane), plane, lat_of(headers, tag + ' '), tag)
        positions = {str(uid): index for index, uid in enumerate(studies)}
        missing = [uid for uid in expected_ids if uid not in positions]
        if missing:
            raise RuntimeError(f'{len(missing)} studies absent from {tag}')
        order = _rad_np.asarray([positions[uid] for uid in expected_ids], dtype=_rad_np.int64)
        pixels, masks = (pixels[order], masks[order])
        tokens = int(_rad_np.repeat(masks[:, :, None], CACHE_SLICES, axis=2).sum())
        if tokens < int(threshold * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f'insufficient slices for {tag}: {tokens}')
        return (pixels, masks)
    public_slots = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True)]
    pixels, masks = cache(public_slots, 10000.0, 'test-e10', 0.85)
    encoder_path = _rad_find_file('ResNet50.pt', _RAD_ENCODER_SHA256)
    encoder = _RadEncoder()
    encoder.load_state_dict(_rad_torch.load(encoder_path, map_location='cpu', weights_only=True), strict=True)
    encoder.eval().to(device)
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    if _rad_torch.cuda.device_count() > 1:
        encoder = _rad_nn.DataParallel(encoder, device_ids=list(range(_rad_torch.cuda.device_count())))
    reference_heads, _ = _rad_load_public_heads(device, _RAD_REFERENCE_HEADS_SHA256)
    features, token_mask = _rad_encode(encoder, pixels, masks, device)
    reference_predictions = [_rad_predict_head(head, features, token_mask, device) for head in reference_heads]
    reference_probability = _rad_np.mean(_rad_np.stack(reference_predictions), axis=0)
    reference_rank = _rad_rank_columns(reference_probability)
    del reference_predictions, reference_heads
    del reference_probability, features, token_mask, pixels, masks
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()
    globals().update(SLOTS=list(_RAD_E13_SLOTS), N_SLOT=len(_RAD_E13_SLOTS), CACHE_SLICES=_RAD_E13_CACHE_SLICES, IMG=_RAD_E13_IMG, CACHE_IMG=_RAD_E13_IMG, CROP_MM=_RAD_E13_CROP_MM, RULES=dict(RULES_LEGACY))
    e13_heads, _ = _rad_load_e13_heads(device)
    pixels, masks = cache(_RAD_E13_SLOTS, _RAD_E13_CROP_MM, 'test-e13', 0.85)
    features, token_mask = _rad_encode(encoder, pixels, masks, device)
    e13_predictions = [_rad_predict_head(head, features, token_mask, device) for head in e13_heads]
    e13_probability = _rad_np.mean(_rad_np.stack(e13_predictions), axis=0)
    e13_rank = _rad_rank_columns(e13_probability)
    reference_rank = _rad_rank_columns((1.0 - _RAD_E13_MEMBER_WEIGHT) * reference_rank + _RAD_E13_MEMBER_WEIGHT * e13_rank)
    del e13_predictions, e13_probability, e13_rank
    del features, token_mask, pixels, masks
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()
    baseline_rank = _rad_rank_columns(baseline[_RAD_LABELS].to_numpy())
    e10 = baseline.copy()
    for index, target in enumerate(_RAD_LABELS):
        if target not in _RAD_EXCLUDE:
            e10[target] = (1.0 - _RAD_ALPHA) * baseline_rank[:, index] + _RAD_ALPHA * reference_rank[:, index]
    _rad_validate(e10, expected_ids)
    pixels, masks = cache(_RAD_E11_SLOTS, _RAD_E11_CROP_MM, 'test-v48-pass2', 0.55)
    features, token_mask = _rad_encode(encoder, pixels, masks, device)
    pass2_predictions = [_rad_predict_head(head, features, token_mask, device) for head in e13_heads]
    pass2_probability = _rad_np.mean(_rad_np.stack(pass2_predictions), axis=0)
    pass2_rank = _rad_rank_columns(pass2_probability)
    final = e10.copy()
    final[_RAD_LABELS] = (
        (1.0 - _RAD_V48_SECOND_ALPHA)
        * _rad_rank_columns(
            e10[_RAD_LABELS].to_numpy()
        )
        + _RAD_V48_SECOND_ALPHA
        * pass2_rank
    )

    final[_RAD_LABELS] = _rad_rank_columns(
        final[_RAD_LABELS].to_numpy()
    )

    _rad_validate(
        final,
        expected_ids,
    )

    globals()['V18_TRANSFORMER_RAW'] = (
        final.copy()
    )
    globals()['V18_CALIBRATOR_APPLIED'] = False
    globals()['V18_CAL_GATE'] = tuple()

    try:
        calibrated, gate = (
            _v18_calibrate_transformer(
                final,
                baseline_rank,
                reference_rank,
                pass2_rank,
                expected_ids,
            )
        )

        final = calibrated

        globals()[
            'V18_TRANSFORMER_CAL'
        ] = final.copy()

        globals()[
            'V18_CALIBRATOR_APPLIED'
        ] = True

        globals()[
            'V18_CAL_GATE'
        ] = tuple(
            sorted(gate)
        )

        print(
            '[V18] 88-feature transformer '
            'calibration applied to: '
            + ', '.join(
                sorted(gate)
            ),
            flush=True,
        )

    except Exception as exc:
        print(
            '[V18] calibration skipped '
            'safely; raw transformer kept: '
            f'{type(exc).__name__}: {exc}',
            flush=True,
        )

    _rad_validate(
        final,
        expected_ids,
    )

    final.to_csv(
        primary,
        index=False,
    )
_rad_main()


[   72.3s] test-e10 laterality: 1 from the tag, 2 from geometry, 0 unresolved; tag and geometry disagree on 0 (0.0% of the tagged)
[   72.3s] test-e10: cache (3, 3, 8, 224, 224) = 0.0 GB
[   72.3s] test-e10: ordering 8 slot-series (366 slice headers)
[   73.1s] test-e10: ordered 8/8 by geometry (0 kept arbitrary) in 1s
[   73.1s] test-e10: decoding 8 slot-series
[   73.5s]   test-e10 8/8
[   73.5s] test-e10: 8/8 slots filled
RadImageNet asset: /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/resnet-50-radimagenet-marwan/ResNet50.pt
RadImageNet asset: /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt
RadImageNet asset: /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/kernel-sources/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt
[   77.8s] test-e13 laterality: 1 from the tag, 2 from geometry, 0 unresolved; tag and geometry disagree on 0 (0.0% of the tagged)
[

In [6]:
# COATNET_TRANSFORMER_BLEND_V1
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
from concurrent.futures import ThreadPoolExecutor
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
IMG = 336
CROP_MM = 140.0
# 64 slices per study instead of 44, same proportions. Must match the corpus the weights
# were trained on (knee_corpus_v4.py).
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
         ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(s[2] for s in SLOTS)                     # 64
K_EVAL = 62   # every window position the volume holds, not an evenly spaced subset
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [
    {
        "file": "raptor_ft_coatnet_v5_full_swa.pt",
        "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
        "res": 384,
        "w": 1.0,
    },
]

LEGACY_ARM = {
    "file": "raptor_ft_coatnet_v4_full.pt",
    "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
    "res": 384,
    "k_eval": 42,
    "span_lo": 0.06,
    "span_hi": 0.94,
}

V9_SPECIALIST_ARM = {
    "file": "raptor_ft_coatnet_v9_full.pt",
    "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k",
    "res": 384,
    "k_eval": 42,
    "span_lo": 0.06,
    "span_hi": 0.94,
}

PRIMARY_SPAN_LO = 0.02
PRIMARY_SPAN_HI = 0.98


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(
        device,
        non_blocking=True,
    )

    use_cuda = (
        str(device).startswith("cuda")
    )

    def _forward():
        return torch.sigmoid(
            model(x).float()
        )[0].cpu().numpy()

    if use_cuda:
        try:
            with torch.autocast(
                "cuda",
                dtype=torch.float16,
            ):
                return _forward()

        except RuntimeError as error:
            try:
                with torch.cuda.device(device):
                    torch.cuda.empty_cache()
            except Exception:
                pass

            print(
                "[DINOsaur V4.2] "
                f"{device} fp16 retry in fp32: "
                f"{type(error).__name__}",
                flush=True,
            )

            return _forward()

    return _forward()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and len(iop) == 6:
                    r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                    n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps); recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def _fill_variant_volume(
    target_volume,
    offset,
    picks,
    pixel_cache,
    files,
    med_ps,
    mm_crop_resize,
):
    arrays = []
    spacings = []

    for position in picks:
        position = min(
            int(position),
            len(files) - 1,
        )
        file_path, spacing = files[
            position
        ]
        arrays.append(
            pixel_cache.get(
                position
            )
        )
        spacings.append(
            spacing
            if spacing > 0
            else med_ps
        )

    valid = [
        array
        for array in arrays
        if array is not None
    ]

    if valid:
        all_pixels = np.concatenate(
            [
                array.ravel()
                for array in valid
            ]
        )
        low, high = np.percentile(
            all_pixels,
            [2.0, 98.0],
        )
    else:
        low, high = 0.0, 1.0

    for local_index, (
        array,
        spacing,
    ) in enumerate(
        zip(
            arrays,
            spacings,
        )
    ):
        output_index = (
            offset
            + local_index
        )

        if (
            output_index
            >= MAXS
        ):
            break

        if array is None:
            continue

        normalized = np.clip(
            (
                array - low
            )
            / (
                high - low
                + 1e-6
            ),
            0,
            1,
        )

        normalized = mm_crop_resize(
            normalized,
            spacing,
        )

        target_volume[
            output_index
        ] = (
            normalized
            * 255
        ).astype(
            np.uint8
        )


def build_study_pair(
    sid,
    ser_records,
    tsdir,
    reader,
):
    """
    Produce exact MaxSpan and legacy-span volumes while reading every required
    DICOM only once. Both checkpoints keep their own percentile normalization.
    """
    (
        order_and_meta,
        read_px,
        mm_crop_resize,
    ) = reader

    rows = ser_records.get(
        sid,
        [],
    )

    primary_volume = np.zeros(
        (
            MAXS,
            IMG,
            IMG,
        ),
        np.uint8,
    )
    legacy_volume = np.zeros_like(
        primary_volume
    )

    used = set()
    offset = 0

    for plane, fluid, count in SLOTS:
        record = _pick_series_for_slot(
            rows,
            plane,
            fluid,
            used,
        )

        if record is None:
            offset += count
            continue

        used.add(
            record[
                "SeriesInstanceUID"
            ]
        )

        files, med_ps = order_and_meta(
            f"{tsdir}/{sid}/"
            f"{record['SeriesInstanceUID']}"
        )

        if not files:
            offset += count
            continue

        number = len(files)

        primary_low = int(
            number
            * PRIMARY_SPAN_LO
        )
        primary_high = int(
            number
            * PRIMARY_SPAN_HI
        ) - 1
        primary_high = max(
            primary_high,
            primary_low,
        )

        legacy_low = int(
            number
            * float(
                LEGACY_ARM[
                    "span_lo"
                ]
            )
        )
        legacy_high = int(
            number
            * float(
                LEGACY_ARM[
                    "span_hi"
                ]
            )
        ) - 1
        legacy_high = max(
            legacy_high,
            legacy_low,
        )

        if number > 1:
            primary_picks = np.linspace(
                primary_low,
                primary_high,
                count,
            ).round().astype(int)

            legacy_picks = np.linspace(
                legacy_low,
                legacy_high,
                count,
            ).round().astype(int)
        else:
            primary_picks = np.zeros(
                count,
                dtype=int,
            )
            legacy_picks = np.zeros(
                count,
                dtype=int,
            )

        required_positions = sorted(
            set(
                primary_picks.tolist()
                + legacy_picks.tolist()
            )
        )

        pixel_cache = {}

        for position in required_positions:
            position = min(
                int(position),
                number - 1,
            )

            file_path, _ = files[
                position
            ]

            try:
                pixel_cache[
                    position
                ] = read_px(
                    file_path
                )
            except Exception:
                pixel_cache[
                    position
                ] = None

        _fill_variant_volume(
            primary_volume,
            offset,
            primary_picks,
            pixel_cache,
            files,
            med_ps,
            mm_crop_resize,
        )

        _fill_variant_volume(
            legacy_volume,
            offset,
            legacy_picks,
            pixel_cache,
            files,
            med_ps,
            mm_crop_resize,
        )

        offset += count

        if offset >= MAXS:
            break

    primary_mask = (
        primary_volume.reshape(
            MAXS,
            -1,
        ).sum(1)
        > 0
    ).astype(
        np.uint8
    )

    legacy_mask = (
        legacy_volume.reshape(
            MAXS,
            -1,
        ).sum(1)
        > 0
    ).astype(
        np.uint8
    )

    return (
        primary_volume,
        primary_mask,
        legacy_volume,
        legacy_mask,
    )


# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    return str(_RSNA_COMP_ROOT)


def find_weight_file(
    fname,
    required=True,
):
    resolved = find_input_file(
        fname,
        preferred=('raptor-knee-finespacing', 'raptor-knee-maxspan', 'raptor-knee-widedense'),
        required=required,
    )
    return str(resolved) if resolved is not None else None


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    reader = _make_reader()

    number_studies = len(
        test_ids
    )

    primary_predictions = np.full(
        (
            number_studies,
            len(LAB),
        ),
        0.5,
        np.float32,
    )

    legacy_predictions = np.full_like(
        primary_predictions,
        0.5,
    )

    legacy_success = np.zeros(
        number_studies,
        dtype=np.bool_,
    )

    v9_predictions = np.full_like(primary_predictions, 0.5)
    v9_success = np.zeros(number_studies, dtype=np.bool_)

    if (
        torch.cuda.is_available()
        and torch.cuda.device_count() >= 1
    ):
        primary_device = torch.device(
            "cuda:0"
        )
    else:
        primary_device = torch.device(
            "cpu"
        )

    legacy_path = find_weight_file(
        LEGACY_ARM[
            "file"
        ],
        required=False,
    )

    legacy_enabled = (
        legacy_path is not None
        and torch.cuda.is_available()
        and torch.cuda.device_count() >= 2
    )

    v9_path = find_weight_file(V9_SPECIALIST_ARM["file"], required=False)
    v9_enabled = bool(legacy_enabled and v9_path is not None)
    if v9_path is not None:
        print(f"[DINOsaur V4.2] FineSpacing checkpoint: {v9_path}", flush=True)
    else:
        print(
            "[DINOsaur V4.2] FineSpacing checkpoint is not mounted. "
            "Add Input: dreaddevelopment/raptor-knee-finespacing",
            flush=True,
        )
    if v9_path is not None and not legacy_enabled:
        print(
            "[DINOsaur V4.2] FineSpacing requires GPU T4 x2 in this pipeline",
            flush=True,
        )

    primary_path = find_weight_file(
        ARMS[0][
            "file"
        ],
        required=True,
    )

    primary_model, primary_res = load_model(
        primary_path,
        ARMS[0][
            "arch"
        ],
        ARMS[0][
            "res"
        ],
        primary_device,
    )

    print(
        "[DINOsaur V4.2] primary "
        f"{ARMS[0]['file']} "
        f"on {primary_device}",
        flush=True,
    )

    legacy_model = None
    legacy_device = None
    legacy_res = None

    if legacy_enabled:
        legacy_device = torch.device(
            "cuda:1"
        )

        try:
            legacy_model, legacy_res = load_model(
                legacy_path,
                LEGACY_ARM[
                    "arch"
                ],
                LEGACY_ARM[
                    "res"
                ],
                legacy_device,
            )

            print(
                "[DINOsaur V4.2] complement "
                f"{LEGACY_ARM['file']} "
                f"on {legacy_device}",
                flush=True,
            )

        except Exception as error:
            legacy_enabled = False
            legacy_model = None

            print(
                "[DINOsaur V4.2] "
                "legacy checkpoint disabled "
                f"safely: "
                f"{type(error).__name__}: "
                f"{error}",
                flush=True,
            )

    else:
        print(
            "[DINOsaur V4.2] "
            "legacy complement unavailable "
            "or second GPU absent; "
            "exact 0.935 Raptor retained",
            flush=True,
        )

    v9_model = None
    v9_res = None
    if v9_enabled:
        try:
            v9_model, v9_res = load_model(
                v9_path,
                V9_SPECIALIST_ARM["arch"],
                V9_SPECIALIST_ARM["res"],
                legacy_device,
            )
            if int(v9_res) != int(legacy_res):
                raise RuntimeError("v9 and v4 resolutions differ")
            print(
                "[DINOsaur V4.2] FineSpacing specialist "
                f"{V9_SPECIALIST_ARM['file']} on {legacy_device}",
                flush=True,
            )
        except Exception as error:
            v9_enabled = False
            v9_model = None
            print(
                "[DINOsaur V4.2] FineSpacing v9 disabled safely: "
                f"{type(error).__name__}: {error}",
                flush=True,
            )

    def infer_secondary_models(windows):
        legacy_prediction = infer_probs(legacy_model, windows, legacy_device)
        v9_prediction = None
        if v9_enabled:
            try:
                v9_prediction = infer_probs(v9_model, windows, legacy_device)
            except Exception as error:
                print(
                    "[DINOsaur V4.2] FineSpacing v9 study fallback: "
                    f"{type(error).__name__}: {error}",
                    flush=True,
                )
        return legacy_prediction, v9_prediction

    executor = (
        ThreadPoolExecutor(
            max_workers=2
        )
        if legacy_enabled
        else None
    )

    for study_index, study_id in enumerate(
        test_ids
    ):
        try:
            (
                primary_volume,
                primary_mask,
                legacy_volume,
                legacy_mask,
            ) = build_study_pair(
                study_id,
                SER,
                tsdir,
                reader,
            )

            primary_windows = eval_windows(
                primary_volume,
                primary_mask,
                k=K_EVAL,
                res=primary_res,
                norm=NORM,
            )

            if legacy_enabled:
                legacy_windows = eval_windows(
                    legacy_volume,
                    legacy_mask,
                    k=int(
                        LEGACY_ARM[
                            "k_eval"
                        ]
                    ),
                    res=legacy_res,
                    norm=NORM,
                )

                primary_future = executor.submit(
                    infer_probs,
                    primary_model,
                    primary_windows,
                    primary_device,
                )

                legacy_future = executor.submit(
                    infer_secondary_models,
                    legacy_windows,
                )

                primary_prediction = (
                    primary_future.result()
                )

                try:
                    legacy_prediction, v9_prediction = (
                        legacy_future.result()
                    )
                    legacy_success[study_index] = True
                    if v9_prediction is not None:
                        v9_predictions[study_index] = v9_prediction
                        v9_success[study_index] = True
                except Exception as legacy_error:
                    legacy_prediction = (
                        primary_prediction.copy()
                    )
                    v9_prediction = primary_prediction.copy()

                    print(
                        "[DINOsaur V4.2] "
                        f"legacy study "
                        f"{study_index} fallback: "
                        f"{type(legacy_error).__name__}: "
                        f"{legacy_error}",
                        flush=True,
                    )

                del legacy_windows

            else:
                primary_prediction = infer_probs(
                    primary_model,
                    primary_windows,
                    primary_device,
                )
                legacy_prediction = (
                    primary_prediction.copy()
                )
                v9_prediction = primary_prediction.copy()

            primary_predictions[
                study_index
            ] = primary_prediction

            legacy_predictions[
                study_index
            ] = legacy_prediction

            del (
                primary_volume,
                primary_mask,
                legacy_volume,
                legacy_mask,
                primary_windows,
                primary_prediction,
                legacy_prediction,
                v9_prediction,
            )

        except Exception as error:
            print(
                "[DINOsaur V4.2] "
                f"study {study_index} "
                f"{study_id[:16]} FALLBACK "
                f"({type(error).__name__}: "
                f"{error})",
                flush=True,
            )

        if (
            (
                study_index + 1
            )
            % 100
            == 0
            or study_index + 1
            == number_studies
        ):
            print(
                "[DINOsaur V4.2] "
                f"{study_index+1}/"
                f"{number_studies} | "
                f"{time.time()-t0:.0f}s",
                flush=True,
            )

    if executor is not None:
        executor.shutdown(
            wait=True
        )

    del primary_model

    if legacy_model is not None:
        del legacy_model
    if v9_model is not None:
        del v9_model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    primary_rank = rankpct(
        np.clip(
            primary_predictions,
            0,
            1,
        )
    )

    raptor_rank = (
        primary_rank.copy()
    )

    legacy_fraction = float(
        legacy_success.mean()
    ) if legacy_enabled else 0.0

    if (
        legacy_enabled
        and legacy_fraction >= 0.98
    ):
        legacy_rank = rankpct(
            np.clip(
                legacy_predictions,
                0,
                1,
            )
        )

        # The expanded MaxSpan checkpoint's documented largest gains are ACL,
        # both menisci, Lateral OA and Fracture. Keep it almost pure there.
        # Use the previous 0.934 checkpoint only for the remaining findings,
        # where it may restore complementary ordering.
        complement_weight = {
            "MCL": 0.16,
            "Medial OA": 0.10,
            "PF OA": 0.12,
            "Effusion": 0.10,
            "Synovitis": 0.16,
            "Baker's": 0.12,
            "Contusion": 0.12,
        }

        complement_log = []

        for target_index, target in enumerate(
            LAB
        ):
            weight = float(
                complement_weight.get(
                    target,
                    0.0,
                )
            )

            if weight <= 0:
                continue

            correlation = float(
                np.corrcoef(
                    primary_rank[
                        :,
                        target_index,
                    ],
                    legacy_rank[
                        :,
                        target_index,
                    ],
                )[0, 1]
            )

            if not np.isfinite(
                correlation
            ):
                weight = 0.0
            elif correlation > 0.992:
                weight *= 0.50
            elif correlation < 0.65:
                weight *= 0.40

            if weight <= 0:
                continue

            raptor_rank[
                :,
                target_index,
            ] = (
                (
                    1.0
                    - weight
                )
                * primary_rank[
                    :,
                    target_index,
                ]
                + weight
                * legacy_rank[
                    :,
                    target_index,
                ]
            )

            complement_log.append(
                (
                    target,
                    weight,
                    correlation,
                )
            )

        raptor_rank = rankpct(
            raptor_rank
        )

        print(
            "[DINOsaur V4.2] "
            "legacy complement: "
            + "; ".join(
                f"{target}=w{weight:.3f},"
                f"corr={correlation:.3f}"
                for (
                    target,
                    weight,
                    correlation,
                )
                in complement_log
            ),
            flush=True,
        )

    else:
        print(
            "[DINOsaur V4.2] "
            f"legacy success={legacy_fraction:.3f}; "
            "exact primary Raptor used",
            flush=True,
        )

    _raptor_legacy_was_applied = bool(
        legacy_enabled and legacy_fraction >= 0.98
    )
    globals()["_RAPTOR_LEGACY_APPLIED"] = _raptor_legacy_was_applied
    globals()["_RAPTOR_COMPLEMENT_EFFECTIVE_WEIGHTS"] = (
        {target: float(weight) for target, weight, _ in complement_log}
        if _raptor_legacy_was_applied
        else {}
    )

    _raptor_v9_was_applied = bool(
        v9_enabled and float(v9_success.mean()) >= 0.98
    )
    globals()["_RAPTOR_V9_APPLIED"] = _raptor_v9_was_applied
    if _raptor_v9_was_applied:
        v9_rank = rankpct(np.clip(v9_predictions, 0, 1))
        v9_frame = pd.DataFrame(v9_rank.astype(np.float32), columns=LAB)
        v9_frame.insert(0, "StudyInstanceUID", test_ids)
        v9_frame = v9_frame[sub_cols]
        globals()["_RAPTOR_V9_TEST"] = v9_frame
        v9_frame.to_csv("/kaggle/working/submission_raptor_v9.csv", index=False)
        print(
            "[DINOsaur V4.2] FineSpacing v9 exported | "
            f"success={float(v9_success.mean()):.3f}",
            flush=True,
        )
    else:
        globals()["_RAPTOR_V9_TEST"] = None
        print(
            "[DINOsaur V4.2] FineSpacing v9 unavailable; specialist profiles disabled",
            flush=True,
        )

    ranks = raptor_rank
    # ranks already contain the dual-checkpoint Raptor prediction.
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission_coatnet.csv"
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as _coat_exc:
        import traceback as _coat_traceback
        print(f"CoAtNet branch failed; retaining transformer submission: {type(_coat_exc).__name__}: {_coat_exc}", flush=True)
        _coat_traceback.print_exc()



# Hidden-rerun fail-safe final fusion.
from pathlib import Path as _D42Path
import shutil as _d42_shutil

_d42_primary = _D42Path('/kaggle/working/submission.csv')
_d42_backup = _D42Path('/kaggle/working/.d42_transformer_backup.csv')

if _d42_primary.is_file():
    _d42_shutil.copy2(_d42_primary, _d42_backup)

try:
    # Blend two independently validated rank predictors. The default remains the transformer
    # submission if the CoAtNet branch did not complete, so a recoverable branch failure
    # cannot erase a valid competition artifact.
    from pathlib import Path as _BlendPath
    import numpy as _blend_np
    import pandas as _blend_pd
    
    _blend_work = _BlendPath('/kaggle/working')
    _blend_transformer_path = _blend_work / 'submission.csv'
    _blend_coatnet_path = _blend_work / 'submission_coatnet.csv'
    if _blend_coatnet_path.is_file():
        _blend_transformer = _blend_pd.read_csv(_blend_transformer_path, dtype={'StudyInstanceUID': str})
        _blend_coatnet = _blend_pd.read_csv(_blend_coatnet_path, dtype={'StudyInstanceUID': str})
        _blend_labels = [c for c in _blend_transformer.columns if c != 'StudyInstanceUID']
        if _blend_coatnet.columns.tolist() != _blend_transformer.columns.tolist():
            raise RuntimeError('CoAtNet/transformer submission schema mismatch')
        if _blend_coatnet['StudyInstanceUID'].tolist() != _blend_transformer['StudyInstanceUID'].tolist():
            raise RuntimeError('CoAtNet/transformer study order mismatch')
        _blend_tr = _blend_transformer[
            _blend_labels
        ].rank(
            method='average',
            pct=True,
        )
    
        _blend_cr = _blend_coatnet[
            _blend_labels
        ].rank(
            method='average',
            pct=True,
        )
    
        _blend_output = (
            _blend_transformer.copy()
        )
    
        _coatnet_weight = {
            label: 0.50
            for label in _blend_labels
        }
    
        if globals().get(
            'V18_CALIBRATOR_APPLIED',
            False,
        ):
            # The MaxSpan checkpoint's documented 58-study gains are concentrated
            # on these targets. Move far enough to affect ranking, but keep the
            # calibrated transformer as a substantial independent vote.
            _coatnet_weight.update(
                {
                    'ACL': 0.53,
                    'Medial Meniscus': 0.56,
                    'Lateral Meniscus': 0.61,
                    'Lateral OA': 0.55,
                    'Fracture': 0.61,
                }
            )
    
        _base_0935 = {
            label: 0.50
            for label in _blend_labels
        }
        _base_0935.update(
            {
                'Medial Meniscus': 0.52,
                'Lateral Meniscus': 0.54,
                'Fracture': 0.54,
            }
        )
    
        # Correlation only controls risk. It never creates a new target weight.
        for _label in _blend_labels:
            _correlation = float(
                _blend_np.corrcoef(
                    _blend_tr[
                        _label
                    ].to_numpy(
                        _blend_np.float64
                    ),
                    _blend_cr[
                        _label
                    ].to_numpy(
                        _blend_np.float64
                    ),
                )[0, 1]
            )
    
            if not _blend_np.isfinite(
                _correlation
            ):
                _coatnet_weight[
                    _label
                ] = _base_0935[
                    _label
                ]
    
            elif _correlation > 0.992:
                _coatnet_weight[
                    _label
                ] = (
                    0.65
                    * _coatnet_weight[
                        _label
                    ]
                    + 0.35
                    * _base_0935[
                        _label
                    ]
                )
    
            elif _correlation < 0.60:
                _coatnet_weight[
                    _label
                ] = (
                    0.50
                    * _coatnet_weight[
                        _label
                    ]
                    + 0.50
                    * _base_0935[
                        _label
                    ]
                )
    
        for _label in _blend_labels:
            _cw = float(
                _coatnet_weight[
                    _label
                ]
            )
    
            _blend_output[
                _label
            ] = (
                (
                    1.0
                    - _cw
                )
                * _blend_tr[
                    _label
                ]
                + _cw
                * _blend_cr[
                    _label
                ]
            )
    
        _blend_output[
            _blend_labels
        ] = _blend_output[
            _blend_labels
        ].rank(
            method='average',
            pct=True,
        )
    
        print(
            '[DINOsaur V4.2] CoAtNet target weights: '
            + ', '.join(
                f'{label}='
                f'{_coatnet_weight[label]:.2f}'
                for label in _blend_labels
                if (
                    _coatnet_weight[label]
                    != 0.50
                )
            ),
            flush=True,
        )
    
        _blend_values = _blend_output[
            _blend_labels
        ].to_numpy(
            _blend_np.float64
        )
        if not _blend_np.isfinite(_blend_values).all() or _blend_values.min() < 0 or _blend_values.max() > 1:
            raise RuntimeError('invalid blended prediction values')
        _blend_output.to_csv(_blend_transformer_path, index=False)
        print(f'final submission.csv = DINOsaur V4.2 dual-checkpoint target fusion; {_blend_output.shape}', flush=True)
    else:
        print('CoAtNet output unavailable; submission.csv remains the validated transformer ensemble', flush=True)
    
    # V18 output hygiene.
    for _v18_temp in (
        _blend_work / 'submission_coatnet.csv',
        _blend_work / 'submission_transformer_0920.csv',
    ):
        try:
            if _v18_temp.is_file():
                _v18_temp.unlink()
        except OSError:
            pass

except Exception as _d42_error:
    import traceback as _d42_traceback
    print(
        '[DINOsaur V4.2] final fusion failed; '
        'restoring calibrated transformer submission: '
        f'{type(_d42_error).__name__}: {_d42_error}',
        flush=True,
    )
    _d42_traceback.print_exc()
    if _d42_backup.is_file():
        _d42_shutil.copy2(_d42_backup, _d42_primary)

finally:
    try:
        if _d42_backup.is_file():
            _d42_backup.unlink()
    except OSError:
        pass

if not _d42_primary.is_file():
    raise RuntimeError('submission.csv missing after V4.2 fail-safe')


device cuda | gpus 2 | torch 2.10.0+cu128
test root: /kaggle/input/competitions/rsna-knee-abnormality-detection | series dir: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
test studies 3 | test series 15
[DINOsaur V4.2] FineSpacing checkpoint is not mounted. Add Input: dreaddevelopment/raptor-knee-finespacing
[DINOsaur V4.2] primary raptor_ft_coatnet_v5_full_swa.pt on cuda:0
[DINOsaur V4.2] complement raptor_ft_coatnet_v4_full.pt on cuda:1
[DINOsaur V4.2] 3/3 | 49s
[DINOsaur V4.2] legacy complement: MCL=w0.080,corr=1.000; Medial OA=w0.040,corr=0.500; PF OA=w0.060,corr=1.000; Effusion=w0.050,corr=1.000; Synovitis=w0.080,corr=1.000; Baker's=w0.060,corr=1.000; Contusion=w0.048,corr=0.500
[DINOsaur V4.2] FineSpacing v9 unavailable; specialist profiles disabled
wrote /kaggle/working/submission_coatnet.csv | 3 rows x 13 cols
                                                StudyInstanceUID  ACL  MCL  Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA  PF OA  Eff

## Public-LB exact-proxy pipeline

This section intentionally tunes the final Transformer/CoAtNet target blend on the 58 labelled
studies. Its purpose is to probe the public leaderboard, not to estimate private-test quality.

The original DINOsaur V10 output is preserved as `submission_v10_exact.csv`. The tuner uses
cross-fitted DINOv2 and RadImageNet predictions from the attached reproduction bundle, runs both
Raptor v5 MaxSpan and v4 legacy checkpoints on the gold studies, and reproduces the exact legacy
complement used on hidden test. A third Raptor FineSpacing v9 checkpoint is evaluated from the
same legacy windows and is added only as a low-weight specialist for Lateral Meniscus, Lateral OA,
and Fracture. The gold-58 gate is diagnostic; the fixed specialist profile does not optimize its
weights on those 58 rows.

`submission.csv` is set to the selected profile below. Every alternative remains in
`/kaggle/working` for diagnostics and later notebook versions.


In [7]:
"""Public-LB-oriented target blend tuner for the RSNA DINOsaur V10 notebook.

This module is embedded into ``notebooks/rsna_public_lb_tuner.ipynb``.  It is
deliberately an aggressive leaderboard experiment, not an unbiased estimate of
private-test performance.
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd


TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

RAPTOR_COMPLEMENT_WEIGHTS = {
    "MCL": 0.16,
    "Medial OA": 0.10,
    "PF OA": 0.12,
    "Effusion": 0.10,
    "Synovitis": 0.16,
    "Baker's": 0.12,
    "Contusion": 0.12,
}

RAPTOR_065_TARGETS = {
    "ACL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Lateral OA",
    "Fracture",
}

# Fixed before looking at gold-58 labels. The target set follows the FineSpacing model's
# intended role and the weights stay small because public quad-Raptor blends were not stronger
# as global replacements for the v5+v4 control.
V9_SPECIALIST_WEIGHTS = {
    "Lateral Meniscus": 0.12,
    "Lateral OA": 0.10,
    "Fracture": 0.08,
}

CLINICAL_NEIGHBORS = {
    "ACL": ("Contusion", "Lateral Meniscus"),
    "MCL": ("ACL", "Contusion"),
    "Medial Meniscus": ("Medial OA", "Lateral Meniscus"),
    "Lateral Meniscus": ("ACL", "Contusion", "Medial Meniscus"),
    "Lateral OA": ("Medial OA", "PF OA", "Lateral Meniscus"),
    "Synovitis": ("Effusion", "Baker's"),
    "Fracture": ("Contusion", "Effusion"),
}

CLINICAL_MODERATE = {
    "ACL": -0.10,
    "MCL": -0.10,
    "Medial Meniscus": -0.10,
    "Lateral Meniscus": -0.075,
    "Lateral OA": 0.10,
    "Synovitis": 0.10,
    "Fracture": 0.05,
}

CLINICAL_AGGRESSIVE = {
    "ACL": -0.20,
    "MCL": -0.20,
    "Medial Meniscus": -0.20,
    "Lateral Meniscus": -0.10,
    "Lateral OA": 0.20,
    "Synovitis": 0.175,
    "Fracture": 0.05,
}


def percentile_ranks(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError(f"rank input must be 2D, got {values.shape}")
    if not np.isfinite(values).all():
        raise ValueError("rank input contains non-finite values")
    return pd.DataFrame(values).rank(method="average", pct=True).to_numpy(np.float64)


def column_auc(y, p):
    y = np.asarray(y)
    p = np.asarray(p, dtype=np.float64)
    if np.unique(y).size < 2:
        return np.nan
    positive = y == 1
    n_positive = int(positive.sum())
    n_negative = int((~positive).sum())
    ranks = pd.Series(p).rank(method="average").to_numpy(np.float64)
    mann_whitney = ranks[positive].sum() - n_positive * (n_positive + 1) / 2.0
    return float(mann_whitney / (n_positive * n_negative))


def macro_auc(y, p):
    scores = [column_auc(y[:, j], p[:, j]) for j in range(y.shape[1])]
    finite = [score for score in scores if np.isfinite(score)]
    return float(np.mean(finite)) if finite else np.nan


def rank_blend(transformer_values, raptor_values, weights):
    transformer_rank = percentile_ranks(transformer_values)
    raptor_rank = percentile_ranks(raptor_values)
    weights = np.asarray(weights, dtype=np.float64).reshape(1, -1)
    if weights.shape[1] != transformer_rank.shape[1]:
        raise ValueError("weight count does not match prediction columns")
    mixed = (1.0 - weights) * transformer_rank + weights * raptor_rank
    return percentile_ranks(mixed)


def combine_raptor_components(primary_values, legacy_values, effective_weights=None):
    """Reproduce the hidden-test v5 MaxSpan + v4 legacy rank complement."""
    primary_rank = percentile_ranks(primary_values)
    if legacy_values is None:
        return primary_rank, {}
    legacy_rank = percentile_ranks(legacy_values)
    output = primary_rank.copy()
    applied = {}
    supplied = dict(effective_weights or {})
    for j, target in enumerate(TARGETS):
        if effective_weights is not None:
            weight = float(supplied.get(target, 0.0))
            correlation = float(np.corrcoef(primary_rank[:, j], legacy_rank[:, j])[0, 1])
        else:
            weight = float(RAPTOR_COMPLEMENT_WEIGHTS.get(target, 0.0))
            correlation = float(np.corrcoef(primary_rank[:, j], legacy_rank[:, j])[0, 1])
            if not np.isfinite(correlation):
                weight = 0.0
            elif correlation > 0.992:
                weight *= 0.50
            elif correlation < 0.65:
                weight *= 0.40
        if weight <= 0.0:
            continue
        output[:, j] = (1.0 - weight) * primary_rank[:, j] + weight * legacy_rank[:, j]
        applied[target] = {"weight": weight, "gold_correlation": correlation}
    return percentile_ranks(output), applied


def apply_clinical_residual(values, alphas):
    """Apply all target corrections from the same unmodified rank matrix."""
    base = percentile_ranks(values)
    corrected = base.copy()
    target_index = {target: j for j, target in enumerate(TARGETS)}
    for target, alpha in dict(alphas).items():
        neighbors = CLINICAL_NEIGHBORS[target]
        neighbor_index = [target_index[name] for name in neighbors]
        corrected[:, target_index[target]] = (
            base[:, target_index[target]]
            + float(alpha) * base[:, neighbor_index].mean(axis=1)
        )
    return percentile_ranks(corrected)


def blend_specialist(base_values, specialist_values, alphas):
    """Inject a rank-model specialist into selected targets only."""
    base_rank = percentile_ranks(base_values)
    specialist_rank = percentile_ranks(specialist_values)
    if base_rank.shape != specialist_rank.shape:
        raise ValueError(
            f"specialist shape mismatch: base={base_rank.shape}, v9={specialist_rank.shape}"
        )
    output = base_rank.copy()
    for j, target in enumerate(TARGETS):
        alpha = float(dict(alphas).get(target, 0.0))
        if not 0.0 <= alpha <= 1.0:
            raise ValueError(f"invalid specialist alpha for {target}: {alpha}")
        if alpha > 0.0:
            output[:, j] = (
                (1.0 - alpha) * base_rank[:, j]
                + alpha * specialist_rank[:, j]
            )
    return percentile_ranks(output)


def fit_v9_gold_gate(y_true, base_values, v9_values, n_boot=1000, seed=20260828):
    """Gate fixed v9 alphas; this remains an intentionally optimistic gold diagnostic."""
    y_true = np.asarray(y_true, dtype=np.int64)
    base_rank = percentile_ranks(base_values)
    v9_rank = percentile_ranks(v9_values)
    if y_true.shape != base_rank.shape or base_rank.shape != v9_rank.shape:
        raise ValueError(
            f"v9 gate shape mismatch: y={y_true.shape}, base={base_rank.shape}, v9={v9_rank.shape}"
        )

    gated = {}
    rows = []
    for j, target in enumerate(TARGETS):
        alpha = float(V9_SPECIALIST_WEIGHTS.get(target, 0.0))
        candidate = (1.0 - alpha) * base_rank[:, j] + alpha * v9_rank[:, j]
        base_auc = column_auc(y_true[:, j], base_rank[:, j])
        candidate_auc = column_auc(y_true[:, j], candidate)
        delta = float(candidate_auc - base_auc)
        correlation = float(np.corrcoef(base_rank[:, j], v9_rank[:, j])[0, 1])
        win_rate, q10, q90 = _bootstrap_win_rate(
            y_true[:, j],
            base_rank[:, j],
            candidate,
            n_boot=n_boot,
            seed=seed + j,
        )
        accepted = bool(
            alpha > 0.0
            and delta >= 0.002
            and win_rate >= 0.65
            and np.isfinite(correlation)
            and correlation <= 0.985
        )
        if accepted:
            gated[target] = alpha
        rows.append(
            {
                "target": target,
                "n_positive": int((y_true[:, j] == 1).sum()),
                "fixed_alpha": alpha,
                "base_auc": base_auc,
                "v9_auc": column_auc(y_true[:, j], v9_rank[:, j]),
                "candidate_auc": candidate_auc,
                "gold_delta": delta,
                "bootstrap_win_rate": win_rate,
                "bootstrap_delta_q10": q10,
                "bootstrap_delta_q90": q90,
                "base_v9_rank_correlation": correlation,
                "gold_gate_accepted": accepted,
                "gated_alpha": alpha if accepted else 0.0,
            }
        )
    return gated, pd.DataFrame(rows)


def _stratified_bootstrap_indices(y, rng):
    positive = np.flatnonzero(y == 1)
    negative = np.flatnonzero(y == 0)
    if not len(positive) or not len(negative):
        return None
    return np.concatenate(
        [
            rng.choice(positive, size=len(positive), replace=True),
            rng.choice(negative, size=len(negative), replace=True),
        ]
    )


def _bootstrap_win_rate(y, default_pred, candidate_pred, n_boot, seed):
    rng = np.random.default_rng(seed)
    wins = 0
    valid = 0
    deltas = []
    for _ in range(int(n_boot)):
        index = _stratified_bootstrap_indices(y, rng)
        if index is None:
            break
        default_auc = column_auc(y[index], default_pred[index])
        candidate_auc = column_auc(y[index], candidate_pred[index])
        if not np.isfinite(default_auc) or not np.isfinite(candidate_auc):
            continue
        delta = candidate_auc - default_auc
        deltas.append(delta)
        wins += int(delta > 0)
        valid += 1
    if not valid:
        return 0.0, np.nan, np.nan
    return (
        wins / valid,
        float(np.quantile(deltas, 0.10)),
        float(np.quantile(deltas, 0.90)),
    )


def _bootstrap_optimal_weights(y, transformer, raptor, default_weight, grid, n_boot, seed):
    """Refit the grid optimum in every stratified bootstrap sample."""
    rng = np.random.default_rng(seed)
    selected = []
    for _ in range(int(n_boot)):
        index = _stratified_bootstrap_indices(y, rng)
        if index is None:
            break
        candidates = []
        for weight in grid:
            pred = (1.0 - weight) * transformer[index] + weight * raptor[index]
            score = column_auc(y[index], pred)
            candidates.append((score, -abs(float(weight) - default_weight), float(weight)))
        candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
        if np.isfinite(candidates[0][0]):
            selected.append(candidates[0][2])
    if not selected:
        return np.asarray([], dtype=np.float64)
    return np.asarray(selected, dtype=np.float64)


def _grid_mode(values, default_weight):
    if not len(values):
        return float(default_weight)
    unique, counts = np.unique(np.asarray(values, dtype=np.float64), return_counts=True)
    best_count = counts.max()
    candidates = unique[counts == best_count]
    return float(candidates[np.argmin(np.abs(candidates - default_weight))])


def fit_target_weights(
    y_true,
    transformer_proxy,
    raptor_proxy,
    default_weights,
    grid=None,
    n_boot=1000,
    seed=20260827,
):
    """Fit one CoAtNet weight per target on all 58 gold rows.

    This is intentionally a public-LB tuning routine.  Bootstrap statistics are
    diagnostics and shrinkage controls; they do not make gold-58 an independent
    validation set.
    """
    if grid is None:
        grid = np.round(np.arange(0.25, 0.801, 0.05), 2)
    grid = np.asarray(grid, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.int64)
    tr = percentile_ranks(transformer_proxy)
    cr = percentile_ranks(raptor_proxy)
    defaults = np.asarray(default_weights, dtype=np.float64)
    if y_true.shape != tr.shape or tr.shape != cr.shape:
        raise ValueError(f"shape mismatch: y={y_true.shape}, tr={tr.shape}, cr={cr.shape}")

    rows = []
    profiles = {
        "direction_005": defaults.copy(),
        "bootstrap_shrunk": defaults.copy(),
        "raptor065": defaults.copy(),
        "hybrid_public": defaults.copy(),
        "aggressive": defaults.copy(),
    }

    for j, target in enumerate(TARGETS):
        y = y_true[:, j]
        default_weight = float(defaults[j])
        default_pred = (1.0 - default_weight) * tr[:, j] + default_weight * cr[:, j]
        default_auc = column_auc(y, default_pred)

        candidates = []
        for weight in grid:
            pred = (1.0 - weight) * tr[:, j] + weight * cr[:, j]
            score = column_auc(y, pred)
            candidates.append((score, -abs(float(weight) - default_weight), float(weight), pred))
        candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
        best_auc, _, best_weight, best_pred = candidates[0]
        delta = float(best_auc - default_auc)
        win_rate, boot_q10, boot_q90 = _bootstrap_win_rate(
            y,
            default_pred,
            best_pred,
            n_boot=n_boot,
            seed=seed + j,
        )
        bootstrap_optima = _bootstrap_optimal_weights(
            y,
            tr[:, j],
            cr[:, j],
            default_weight,
            grid,
            n_boot=n_boot,
            seed=seed + 1000 + j,
        )
        bootstrap_median = (
            float(np.median(bootstrap_optima)) if len(bootstrap_optima) else default_weight
        )
        bootstrap_mode = _grid_mode(bootstrap_optima, default_weight)
        if not len(bootstrap_optima):
            direction_probability = 0.0
        elif best_weight > default_weight:
            direction_probability = float(np.mean(bootstrap_optima > default_weight))
        elif best_weight < default_weight:
            direction_probability = float(np.mean(bootstrap_optima < default_weight))
        else:
            direction_probability = 0.0

        direction_weight = default_weight
        if delta >= 0.002 and direction_probability >= 0.70:
            direction_weight = default_weight + 0.05 * np.sign(best_weight - default_weight)

        shrunk = default_weight
        if delta >= 0.002 and direction_probability >= 0.60:
            anchor = 0.5 * bootstrap_median + 0.5 * bootstrap_mode
            shift = np.clip(0.60 * (anchor - default_weight), -0.10, 0.10)
            shrunk = default_weight + shift

        raptor065 = default_weight
        if target in RAPTOR_065_TARGETS:
            raptor065 = 0.65

        hybrid_public = shrunk
        if (
            target in RAPTOR_065_TARGETS
            and best_weight > default_weight
            and direction_probability >= 0.65
        ):
            hybrid_public = max(hybrid_public, 0.65)

        profiles["direction_005"][j] = np.clip(direction_weight, 0.0, 1.0)
        profiles["bootstrap_shrunk"][j] = np.clip(shrunk, 0.0, 1.0)
        profiles["raptor065"][j] = raptor065
        profiles["hybrid_public"][j] = np.clip(hybrid_public, 0.0, 1.0)
        profiles["aggressive"][j] = best_weight

        rows.append(
            {
                "target": target,
                "n_positive": int((y == 1).sum()),
                "n_negative": int((y == 0).sum()),
                "transformer_proxy_auc": column_auc(y, tr[:, j]),
                "raptor_exact_auc": column_auc(y, cr[:, j]),
                "default_weight": default_weight,
                "default_auc": default_auc,
                "best_weight": best_weight,
                "best_auc": best_auc,
                "gold_delta": delta,
                "bootstrap_win_rate": win_rate,
                "bootstrap_delta_q10": boot_q10,
                "bootstrap_delta_q90": boot_q90,
                "bootstrap_optimum_median": bootstrap_median,
                "bootstrap_optimum_mode": bootstrap_mode,
                "bootstrap_direction_probability": direction_probability,
                "direction_005_weight": direction_weight,
                "bootstrap_shrunk_weight": shrunk,
                "raptor065_weight": raptor065,
                "hybrid_public_weight": hybrid_public,
                "aggressive_weight": best_weight,
            }
        )

    return profiles, pd.DataFrame(rows)


def _validate_prediction_frame(frame, expected_ids, targets=TARGETS):
    required = ["StudyInstanceUID"] + list(targets)
    if frame.columns.tolist() != required:
        raise ValueError(f"submission columns drifted: {frame.columns.tolist()}")
    ids = frame["StudyInstanceUID"].astype(str).tolist()
    if ids != list(expected_ids):
        raise ValueError("submission StudyInstanceUID order drifted")
    values = frame[targets].to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError("submission contains non-finite values")
    if values.min() < 0.0 or values.max() > 1.0:
        raise ValueError("submission predictions fall outside [0, 1]")


def _frame_from_blend(transformer, raptor, weights):
    expected_ids = transformer["StudyInstanceUID"].astype(str).tolist()
    if raptor["StudyInstanceUID"].astype(str).tolist() != expected_ids:
        raise ValueError("transformer/CoAtNet row order mismatch")
    output = transformer[["StudyInstanceUID"]].copy()
    output[TARGETS] = rank_blend(
        transformer[TARGETS].to_numpy(np.float64),
        raptor[TARGETS].to_numpy(np.float64),
        weights,
    )
    _validate_prediction_frame(output, expected_ids)
    return output


def _frame_with_clinical(frame, alphas):
    expected_ids = frame["StudyInstanceUID"].astype(str).tolist()
    output = frame[["StudyInstanceUID"]].copy()
    output[TARGETS] = apply_clinical_residual(
        frame[TARGETS].to_numpy(np.float64),
        alphas,
    )
    _validate_prediction_frame(output, expected_ids)
    return output


def _frame_with_specialist(frame, specialist, alphas):
    expected_ids = frame["StudyInstanceUID"].astype(str).tolist()
    if specialist["StudyInstanceUID"].astype(str).tolist() != expected_ids:
        raise ValueError("base/FineSpacing v9 row order mismatch")
    output = frame[["StudyInstanceUID"]].copy()
    output[TARGETS] = blend_specialist(
        frame[TARGETS].to_numpy(np.float64),
        specialist[TARGETS].to_numpy(np.float64),
        alphas,
    )
    _validate_prediction_frame(output, expected_ids)
    return output


def _default_v10_weights(transformer, raptor, calibrator_applied):
    weights = {target: 0.50 for target in TARGETS}
    if calibrator_applied:
        weights.update(
            {
                "ACL": 0.53,
                "Medial Meniscus": 0.56,
                "Lateral Meniscus": 0.61,
                "Lateral OA": 0.55,
                "Fracture": 0.61,
            }
        )
    base_0935 = {target: 0.50 for target in TARGETS}
    base_0935.update(
        {
            "Medial Meniscus": 0.52,
            "Lateral Meniscus": 0.54,
            "Fracture": 0.54,
        }
    )
    tr = percentile_ranks(transformer[TARGETS].to_numpy(np.float64))
    cr = percentile_ranks(raptor[TARGETS].to_numpy(np.float64))
    for j, target in enumerate(TARGETS):
        corr = float(np.corrcoef(tr[:, j], cr[:, j])[0, 1])
        if not np.isfinite(corr):
            weights[target] = base_0935[target]
        elif corr > 0.992:
            weights[target] = 0.65 * weights[target] + 0.35 * base_0935[target]
        elif corr < 0.60:
            weights[target] = 0.50 * weights[target] + 0.50 * base_0935[target]
    return np.asarray([weights[target] for target in TARGETS], dtype=np.float64)


def _read_gold(root):
    train = pd.read_csv(root / "train.csv", dtype={"StudyInstanceUID": str})
    gold = train.loc[train[TARGETS].notna().all(axis=1)].copy()
    if len(gold) != 58:
        raise ValueError(f"expected 58 gold studies, found {len(gold)}")
    ids = gold["StudyInstanceUID"].astype(str).tolist()
    y_true = gold[TARGETS].to_numpy(np.int64)
    return gold, ids, y_true


def _artifact_candidates(input_root, filename):
    """Search attached datasets without descending into competition DICOM trees."""
    shared_resolver = globals().get("find_input_files")
    if callable(shared_resolver):
        return list(shared_resolver(filename, input_root=input_root))

    input_root = Path(input_root)
    data_roots = []
    if input_root.is_dir():
        for child in input_root.iterdir():
            if not child.is_dir() or child.name.lower() in {"competitions", "models"}:
                continue
            if child.name.lower() == "datasets":
                for owner in child.iterdir():
                    if owner.is_dir():
                        data_roots.extend(path for path in owner.iterdir() if path.is_dir())
            else:
                is_competition = (child / "train.csv").is_file() and (child / "test.csv").is_file()
                if not is_competition:
                    data_roots.append(child)
    hits = []
    for data_root in data_roots:
        hits.extend(path for path in data_root.rglob(filename) if path.is_file())
    return sorted({str(path.resolve()): path for path in hits}.values(), key=str)


def _discover_dinov2_oof(input_root, gold_ids):
    valid = []
    required = {"ids", "pred", "targets", "gold_mask"}
    for path in _artifact_candidates(input_root, "oof.npz"):
        try:
            with np.load(path, allow_pickle=True) as payload:
                if not required.issubset(payload.files):
                    continue
                ids = payload["ids"].astype(str)
                pred = np.asarray(payload["pred"], dtype=np.float64)
                targets = payload["targets"].astype(str).tolist()
                gold_mask = np.asarray(payload["gold_mask"], dtype=bool)
                if pred.shape != (len(ids), len(targets)) or set(targets) != set(TARGETS):
                    continue
                positions = {uid: i for i, uid in enumerate(ids)}
                if not set(gold_ids).issubset(positions):
                    continue
                order = np.asarray([positions[uid] for uid in gold_ids], dtype=np.int64)
                if not gold_mask[order].all():
                    continue
                columns = [targets.index(target) for target in TARGETS]
                values = pred[order][:, columns]
                if np.isfinite(values).all():
                    valid.append((path, values, len(ids)))
        except Exception:
            continue
    if not valid:
        raise FileNotFoundError("could not find the rsna-knee-weights/oof.npz artifact")
    valid.sort(key=lambda item: (item[2] == 4407, "rsna-knee-weights" in str(item[0])), reverse=True)
    path, values, _ = valid[0]
    print(f"DINOv2 OOF: {path}")
    return values, path


def _discover_rad_oof(input_root, gold_ids):
    valid = []
    paths = _artifact_candidates(input_root, "v52_oof.csv")
    paths += _artifact_candidates(input_root, "v52_e11_oof.csv")
    for path in paths:
        try:
            frame = pd.read_csv(path, dtype={"StudyInstanceUID": str})
            if not {"StudyInstanceUID", *TARGETS}.issubset(frame.columns):
                continue
            positions = {uid: i for i, uid in enumerate(frame["StudyInstanceUID"].astype(str))}
            if not set(gold_ids).issubset(positions):
                continue
            order = [positions[uid] for uid in gold_ids]
            values = frame.iloc[order][TARGETS].to_numpy(np.float64)
            if not np.isfinite(values).all():
                continue
            preferred = "rsna-knee-e9-radimagenet-heads-v15" in str(path)
            gold_flag = int(frame.get("is_gold", pd.Series(dtype=int)).sum()) == 58
            valid.append((preferred, gold_flag, len(frame), path, values))
        except Exception:
            continue
    if not valid:
        raise FileNotFoundError("could not find a compatible RadImageNet OOF CSV")
    valid.sort(key=lambda item: (item[0], item[1], item[2] == 4407), reverse=True)
    _, _, _, path, values = valid[0]
    print(f"RadImageNet OOF: {path}")
    return values, path


def _transformer_gold_proxy(dino_oof, rad_oof, excluded=None):
    excluded = set(excluded or {"Baker's", "Fracture"})
    dino_rank = percentile_ranks(dino_oof)
    rad_rank = percentile_ranks(rad_oof)
    proxy = dino_rank.copy()
    for j, target in enumerate(TARGETS):
        if target not in excluded:
            proxy[:, j] = 0.50 * dino_rank[:, j] + 0.50 * rad_rank[:, j]
    return percentile_ranks(proxy)


def _infer_gold_raptor_exact(namespace, gold_ids, root):
    required = [
        "torch",
        "_make_reader",
        "find_weight_file",
        "load_model",
        "build_study_pair",
        "eval_windows",
        "infer_probs",
        "ARMS",
        "LEGACY_ARM",
        "V9_SPECIALIST_ARM",
        "K_EVAL",
        "NORM",
    ]
    missing = [name for name in required if name not in namespace]
    if missing:
        raise RuntimeError(f"baseline CoAtNet globals missing: {missing}")

    torch = namespace["torch"]
    train_series = pd.read_csv(
        root / "train_series.csv",
        dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
    )
    series_by_study = {
        uid: group.to_dict("records")
        for uid, group in train_series.groupby("StudyInstanceUID", sort=False)
    }
    if not set(gold_ids).issubset(series_by_study):
        raise RuntimeError("one or more gold studies are absent from train_series.csv")

    arm = namespace["ARMS"][0]
    legacy_arm = namespace["LEGACY_ARM"]
    primary_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    reader = namespace["_make_reader"]()
    weight_path = namespace["find_weight_file"](arm["file"], required=True)
    primary_model, primary_resolution = namespace["load_model"](
        weight_path,
        arm["arch"],
        arm["res"],
        primary_device,
    )

    legacy_path = namespace["find_weight_file"](legacy_arm["file"], required=False)
    test_legacy_applied = namespace.get("_RAPTOR_LEGACY_APPLIED")
    if test_legacy_applied is None:
        test_legacy_applied = bool(
            legacy_path is not None and torch.cuda.is_available() and torch.cuda.device_count() >= 2
        )
        print("Raptor legacy marker absent; inferred state from checkpoint/GPU availability")
    test_effective_weights = namespace.get("_RAPTOR_COMPLEMENT_EFFECTIVE_WEIGHTS")

    legacy_model = None
    legacy_device = None
    legacy_resolution = None
    if test_legacy_applied:
        if legacy_path is None:
            raise RuntimeError("test used the legacy Raptor complement but its checkpoint is absent")
        if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
            legacy_device = torch.device("cuda:1")
        else:
            legacy_device = primary_device
        legacy_model, legacy_resolution = namespace["load_model"](
            legacy_path,
            legacy_arm["arch"],
            legacy_arm["res"],
            legacy_device,
        )

    test_v9_applied = bool(namespace.get("_RAPTOR_V9_APPLIED", False))
    v9_arm = namespace["V9_SPECIALIST_ARM"]
    v9_model = None
    v9_resolution = None
    if test_v9_applied:
        v9_path = namespace["find_weight_file"](v9_arm["file"], required=True)
        if legacy_device is None:
            legacy_device = (
                torch.device("cuda:1")
                if torch.cuda.is_available() and torch.cuda.device_count() >= 2
                else primary_device
            )
        v9_model, v9_resolution = namespace["load_model"](
            v9_path,
            v9_arm["arch"],
            v9_arm["res"],
            legacy_device,
        )
        if legacy_resolution is not None and int(v9_resolution) != int(legacy_resolution):
            raise RuntimeError("v9 and v4 Raptor resolutions differ")

    primary_predictions = np.zeros((len(gold_ids), len(TARGETS)), dtype=np.float32)
    legacy_predictions = (
        np.zeros_like(primary_predictions) if test_legacy_applied else None
    )
    v9_predictions = np.zeros_like(primary_predictions) if test_v9_applied else None

    def infer_secondary(windows):
        legacy_prediction = namespace["infer_probs"](
            legacy_model, windows, legacy_device
        )
        v9_prediction = None
        if v9_model is not None:
            v9_prediction = namespace["infer_probs"](
                v9_model, windows, legacy_device
            )
        return legacy_prediction, v9_prediction

    failures = []
    for index, study_id in enumerate(gold_ids):
        try:
            primary_volume, primary_mask, legacy_volume, legacy_mask = namespace["build_study_pair"](
                study_id,
                series_by_study,
                root / "train_series",
                reader,
            )
            primary_windows = namespace["eval_windows"](
                primary_volume,
                primary_mask,
                k=int(namespace["K_EVAL"]),
                res=primary_resolution,
                norm=namespace["NORM"],
            )
            if legacy_model is None:
                primary_predictions[index] = namespace["infer_probs"](
                    primary_model, primary_windows, primary_device
                )
            else:
                legacy_windows = namespace["eval_windows"](
                    legacy_volume,
                    legacy_mask,
                    k=int(legacy_arm["k_eval"]),
                    res=legacy_resolution,
                    norm=namespace["NORM"],
                )
                if primary_device != legacy_device:
                    from concurrent.futures import ThreadPoolExecutor

                    with ThreadPoolExecutor(max_workers=2) as executor:
                        primary_future = executor.submit(
                            namespace["infer_probs"],
                            primary_model,
                            primary_windows,
                            primary_device,
                        )
                        legacy_future = executor.submit(infer_secondary, legacy_windows)
                        primary_predictions[index] = primary_future.result()
                        legacy_prediction, v9_prediction = legacy_future.result()
                        legacy_predictions[index] = legacy_prediction
                        if v9_prediction is not None:
                            v9_predictions[index] = v9_prediction
                else:
                    primary_predictions[index] = namespace["infer_probs"](
                        primary_model, primary_windows, primary_device
                    )
                    legacy_predictions[index] = namespace["infer_probs"](
                        legacy_model, legacy_windows, legacy_device
                    )
                    if v9_model is not None:
                        v9_predictions[index] = namespace["infer_probs"](
                            v9_model, legacy_windows, legacy_device
                        )
        except Exception as exc:
            failures.append((study_id, type(exc).__name__, str(exc)))
        if (index + 1) % 10 == 0 or index + 1 == len(gold_ids):
            print(f"Exact Raptor gold: {index + 1}/{len(gold_ids)}")

    del primary_model
    if legacy_model is not None:
        del legacy_model
    if v9_model is not None:
        del v9_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if failures:
        preview = "; ".join(f"{uid}: {kind}" for uid, kind, _ in failures[:5])
        raise RuntimeError(f"Raptor gold inference failed for {len(failures)} studies: {preview}")
    if not np.isfinite(primary_predictions).all():
        raise RuntimeError("primary Raptor gold predictions contain non-finite values")
    if legacy_predictions is not None and not np.isfinite(legacy_predictions).all():
        raise RuntimeError("legacy Raptor gold predictions contain non-finite values")
    if v9_predictions is not None and not np.isfinite(v9_predictions).all():
        raise RuntimeError("FineSpacing v9 gold predictions contain non-finite values")

    effective = test_effective_weights if test_legacy_applied else {}
    raptor_exact, applied = combine_raptor_components(
        primary_predictions,
        legacy_predictions,
        effective_weights=effective if test_effective_weights is not None else None,
    )
    print(
        "Exact Raptor gold complement: "
        + (", ".join(f"{target}={item['weight']:.3f}" for target, item in applied.items()) or "disabled")
    )
    return {
        "primary": primary_predictions,
        "legacy": legacy_predictions,
        "v9": v9_predictions,
        "exact": raptor_exact,
        "legacy_applied": bool(test_legacy_applied),
        "v9_applied": bool(test_v9_applied),
        "effective_weights": applied,
    }


def run_public_lb_tuner(namespace, final_profile="v9_specialist", n_boot=1000):
    """Run after all original DINOsaur V10 cells have completed."""
    valid_profiles = {
        "direction_005",
        "bootstrap_shrunk",
        "raptor065",
        "clinical_moderate",
        "hybrid_public",
        "aggressive",
        "aggressive_residual",
        "v9_specialist",
        "v9_gold_gated",
    }
    if final_profile not in valid_profiles:
        raise ValueError(f"unknown final profile: {final_profile}")

    work = Path("/kaggle/working")
    input_root = Path("/kaggle/input")
    root = Path(namespace["ROOT"])
    exact_path = work / "submission.csv"
    if not exact_path.is_file():
        raise FileNotFoundError("DINOsaur V10 did not produce submission.csv")
    exact = pd.read_csv(exact_path, dtype={"StudyInstanceUID": str})
    expected_ids = exact["StudyInstanceUID"].astype(str).tolist()
    _validate_prediction_frame(exact, expected_ids)
    exact.to_csv(work / "submission_v10_exact.csv", index=False)

    transformer = namespace.get("_blend_transformer")
    raptor = namespace.get("_blend_coatnet")
    if transformer is None or raptor is None:
        raise RuntimeError(
            "V10 component frames are unavailable. The CoAtNet branch must finish successfully "
            "for target tuning; submission_v10_exact.csv is still valid."
        )
    transformer = transformer[["StudyInstanceUID"] + TARGETS].copy()
    raptor = raptor[["StudyInstanceUID"] + TARGETS].copy()
    _validate_prediction_frame(transformer, expected_ids)
    _validate_prediction_frame(raptor, expected_ids)
    transformer.to_csv(work / "submission_transformer_v18.csv", index=False)
    raptor.to_csv(work / "submission_raptor.csv", index=False)

    v9_test = namespace.get("_RAPTOR_V9_TEST")
    if v9_test is not None:
        v9_test = v9_test[["StudyInstanceUID"] + TARGETS].copy()
        _validate_prediction_frame(v9_test, expected_ids)
        v9_test.to_csv(work / "submission_raptor_v9.csv", index=False)
    elif final_profile in {"v9_specialist", "v9_gold_gated"}:
        exact.to_csv(exact_path, index=False)
        status = pd.DataFrame(
            [
                {
                    "requested_profile": final_profile,
                    "effective_profile": "exact",
                    "reason": "FineSpacing v9 checkpoint unavailable",
                    "required_input": "dreaddevelopment/raptor-knee-finespacing",
                    "required_accelerator": "GPU T4 x2",
                }
            ]
        )
        status.to_csv(work / "submission_profile_status.csv", index=False)
        print(
            "WARNING: FineSpacing v9 is unavailable. "
            "Kept exact DINOsaur V10 in submission.csv so the Kaggle run can succeed."
        )
        print("Add Input: dreaddevelopment/raptor-knee-finespacing | Accelerator: GPU T4 x2")
        return {
            "requested_profile": final_profile,
            "effective_profile": "exact",
            "outputs": {"exact": exact},
            "fallback_reason": "FineSpacing v9 checkpoint unavailable",
        }

    calibrator_applied = bool(namespace.get("V18_CALIBRATOR_APPLIED", False))
    default_weights = _default_v10_weights(transformer, raptor, calibrator_applied)
    recomputed = _frame_from_blend(transformer, raptor, default_weights)
    drift = float(
        np.max(
            np.abs(
                recomputed[TARGETS].to_numpy(np.float64)
                - exact[TARGETS].to_numpy(np.float64)
            )
        )
    )
    print(f"Exact V10 reconstruction max abs drift: {drift:.8f}")
    if drift > 1e-3:
        raise RuntimeError("could not reproduce the exact V10 final blend from component frames")

    gold, gold_ids, y_true = _read_gold(root)
    dino_oof, dino_path = _discover_dinov2_oof(input_root, gold_ids)
    rad_oof, rad_path = _discover_rad_oof(input_root, gold_ids)
    transformer_proxy = _transformer_gold_proxy(
        dino_oof,
        rad_oof,
        excluded=namespace.get("_RAD_EXCLUDE", {"Baker's", "Fracture"}),
    )
    raptor_gold = _infer_gold_raptor_exact(namespace, gold_ids, root)

    gold_default = rank_blend(transformer_proxy, raptor_gold["exact"], default_weights)
    v9_gated_weights = {}
    v9_diagnostics = pd.DataFrame()
    if raptor_gold["v9"] is not None:
        v9_gated_weights, v9_diagnostics = fit_v9_gold_gate(
            y_true,
            gold_default,
            raptor_gold["v9"],
            n_boot=n_boot,
        )
        v9_diagnostics.to_csv(work / "v9_specialist_diagnostics.csv", index=False)

    profiles, diagnostics = fit_target_weights(
        y_true,
        transformer_proxy,
        raptor_gold["exact"],
        default_weights,
        n_boot=n_boot,
    )
    diagnostics.insert(0, "profile_selected", final_profile)
    diagnostics.to_csv(work / "blend_diagnostics.csv", index=False)

    component_frame = gold[["StudyInstanceUID"]].copy()
    for prefix, values in (
        ("dino_oof", dino_oof),
        ("rad_oof", rad_oof),
        ("transformer_proxy", transformer_proxy),
        ("raptor_v5_primary", raptor_gold["primary"]),
        ("raptor_exact", raptor_gold["exact"]),
    ):
        for j, target in enumerate(TARGETS):
            component_frame[f"{prefix}__{target}"] = values[:, j]
    if raptor_gold["legacy"] is not None:
        for j, target in enumerate(TARGETS):
            component_frame[f"raptor_v4_legacy__{target}"] = raptor_gold["legacy"][:, j]
    if raptor_gold["v9"] is not None:
        for j, target in enumerate(TARGETS):
            component_frame[f"raptor_v9_finespacing__{target}"] = raptor_gold["v9"][:, j]
    component_frame.to_csv(work / "gold_component_predictions.csv", index=False)

    complement_rows = []
    for target in TARGETS:
        item = raptor_gold["effective_weights"].get(target, {})
        complement_rows.append(
            {
                "target": target,
                "legacy_applied_on_test": raptor_gold["legacy_applied"],
                "effective_legacy_weight": float(item.get("weight", 0.0)),
                "gold_primary_legacy_correlation": item.get("gold_correlation", np.nan),
            }
        )
    pd.DataFrame(complement_rows).to_csv(work / "raptor_exact_proxy.csv", index=False)

    weight_table = pd.DataFrame({"target": TARGETS})
    weight_table["default"] = default_weights
    for profile, weights in profiles.items():
        weight_table[profile] = weights
    weight_table.to_csv(work / "blend_weights.csv", index=False)

    outputs = {"exact": exact}
    for profile, weights in profiles.items():
        frame = _frame_from_blend(transformer, raptor, weights)
        if profile == "hybrid_public":
            frame = _frame_with_clinical(frame, CLINICAL_MODERATE)
        path = work / f"submission_public_{profile}.csv"
        frame.to_csv(path, index=False)
        outputs[profile] = frame

    clinical_moderate = _frame_with_clinical(
        _frame_from_blend(transformer, raptor, default_weights),
        CLINICAL_MODERATE,
    )
    clinical_moderate.to_csv(work / "submission_public_clinical_moderate.csv", index=False)
    outputs["clinical_moderate"] = clinical_moderate

    aggressive_residual = _frame_with_clinical(
        outputs["aggressive"],
        CLINICAL_AGGRESSIVE,
    )
    aggressive_residual.to_csv(work / "submission_public_aggressive_residual.csv", index=False)
    outputs["aggressive_residual"] = aggressive_residual

    if v9_test is not None:
        v9_specialist = _frame_with_specialist(
            exact,
            v9_test,
            V9_SPECIALIST_WEIGHTS,
        )
        v9_specialist.to_csv(work / "submission_public_v9_specialist.csv", index=False)
        outputs["v9_specialist"] = v9_specialist

        v9_gold_gated = _frame_with_specialist(
            exact,
            v9_test,
            v9_gated_weights,
        )
        v9_gold_gated.to_csv(work / "submission_public_v9_gold_gated.csv", index=False)
        outputs["v9_gold_gated"] = v9_gold_gated

    gold_profile_rows = [
        {
            "profile": "exact_weight_reconstruction",
            "gold_macro_auc": macro_auc(y_true, gold_default),
            "gold_delta_vs_default": 0.0,
            "clinical_residual": "none",
        }
    ]
    for profile, weights in profiles.items():
        gold_values = rank_blend(transformer_proxy, raptor_gold["exact"], weights)
        residual_name = "none"
        if profile == "hybrid_public":
            gold_values = apply_clinical_residual(gold_values, CLINICAL_MODERATE)
            residual_name = "moderate"
        score = macro_auc(y_true, gold_values)
        gold_profile_rows.append(
            {
                "profile": profile,
                "gold_macro_auc": score,
                "gold_delta_vs_default": score - macro_auc(y_true, gold_default),
                "clinical_residual": residual_name,
            }
        )
    for profile, weights, alphas, residual_name in (
        ("clinical_moderate", default_weights, CLINICAL_MODERATE, "moderate"),
        ("aggressive_residual", profiles["aggressive"], CLINICAL_AGGRESSIVE, "aggressive"),
    ):
        gold_values = rank_blend(transformer_proxy, raptor_gold["exact"], weights)
        gold_values = apply_clinical_residual(gold_values, alphas)
        score = macro_auc(y_true, gold_values)
        gold_profile_rows.append(
            {
                "profile": profile,
                "gold_macro_auc": score,
                "gold_delta_vs_default": score - macro_auc(y_true, gold_default),
                "clinical_residual": residual_name,
            }
        )
    if raptor_gold["v9"] is not None:
        for profile, alphas, gate_name in (
            ("v9_specialist", V9_SPECIALIST_WEIGHTS, "fixed_pre_gold"),
            ("v9_gold_gated", v9_gated_weights, "gold_gate"),
        ):
            gold_values = blend_specialist(gold_default, raptor_gold["v9"], alphas)
            score = macro_auc(y_true, gold_values)
            gold_profile_rows.append(
                {
                    "profile": profile,
                    "gold_macro_auc": score,
                    "gold_delta_vs_default": score - macro_auc(y_true, gold_default),
                    "clinical_residual": gate_name,
                }
            )
    gold_profile_metrics = pd.DataFrame(gold_profile_rows).sort_values(
        "gold_macro_auc", ascending=False
    )
    gold_profile_metrics.to_csv(work / "gold_profile_metrics.csv", index=False)

    chosen = outputs[final_profile]
    chosen.to_csv(exact_path, index=False)
    _validate_prediction_frame(chosen, expected_ids)

    print("\nGold-58 target diagnostics")
    print(
        diagnostics[
            [
                "target",
                "n_positive",
                "default_weight",
                "best_weight",
                "gold_delta",
                "bootstrap_win_rate",
                "bootstrap_direction_probability",
                "bootstrap_shrunk_weight",
            ]
        ].to_string(index=False)
    )
    print("\nGold profile ranking (diagnostic only; optimistic)")
    print(gold_profile_metrics.to_string(index=False))
    print(f"\nDINO OOF source: {dino_path}")
    print(f"Rad OOF source:  {rad_path}")
    print(f"Final submission.csv profile: {final_profile}")
    print("Preserved control: submission_v10_exact.csv")
    print(
        "Candidates: v9_specialist, v9_gold_gated, direction_005, bootstrap_shrunk, "
        "raptor065, clinical_moderate, hybrid_public, aggressive, aggressive_residual"
    )
    return {
        "profiles": profiles,
        "diagnostics": diagnostics,
        "outputs": outputs,
        "default_weights": default_weights,
        "gold_profile_metrics": gold_profile_metrics,
        "raptor_gold": raptor_gold,
        "v9_gated_weights": v9_gated_weights,
        "v9_diagnostics": v9_diagnostics,
        "exact_reconstruction_drift": drift,
    }


In [8]:
PUBLIC_TUNER_CFG = {
    "final_profile": FINAL_PROFILE,
    "n_boot": N_BOOTSTRAPS,
}

print(f"Selected public-LB profile: {FINAL_PROFILE} | bootstraps: {N_BOOTSTRAPS}")
PUBLIC_TUNER_RESULT = run_public_lb_tuner(globals(), **PUBLIC_TUNER_CFG)


Selected public-LB profile: clinical_moderate | bootstraps: 1000
Exact V10 reconstruction max abs drift: 0.00000000
DINOv2 OOF: /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-weights/oof.npz
RadImageNet OOF: /kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/rsna-knee-e9-radimagenet-heads-v15/v52_oof.csv
Exact Raptor gold: 10/58
Exact Raptor gold: 20/58
Exact Raptor gold: 30/58
Exact Raptor gold: 40/58
Exact Raptor gold: 50/58
Exact Raptor gold: 58/58
Exact Raptor gold complement: MCL=0.080, Medial OA=0.040, PF OA=0.060, Effusion=0.050, Synovitis=0.080, Baker's=0.060, Contusion=0.048

Gold-58 target diagnostics
          target  n_positive  default_weight  best_weight  gold_delta  bootstrap_win_rate  bootstrap_direction_probability  bootstrap_shrunk_weight
             ACL          24          0.5150         0.80    0.028186               0.994                            0.987                   0.6150
             MCL           9 